### Read the raw job posting data: `data_purge_v1`
- Two files are generated. From the master data, we separate `job description` so that rest of the data are manageable: one with `job description` and the other without `job description`.

In [ ]:
# This script processes job posting data from various sources. It reads data files, cleans and filters the data based on specific criteria (like removing duplicates and filtering by source), and finally extracts and saves job titles and descriptions for further analysis. Requires input data in .dat format and outputs processed data in CSV format.

# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from datetime import datetime
import gc, json, csv, re, os, glob

def read_dat_data(curFile):
    """
    Reads a .dat file containing job posting data and returns a DataFrame.
    The function expects a specific format with predefined column names and uses '@!' as a separator.
    Parameters:
    curFile (str): File path of the .dat file to be read.
    Returns:
    pandas.DataFrame: Contains the job posting data with specified column names.
    """
    colNames = ['招聘主键ID','公司ID','公司名称','城市名称','公司所在区域','工作薪酬','教育要求','工作经历',
                '工作描述','职位名称','工作名称','招聘数量','发布日期','行业名称','数据来源']
    resCSV = pd.read_csv(curFile, header=None, index_col=None, names=colNames,encoding='utf-8',quoting=csv.QUOTE_NONE, sep="@!", error_bad_lines=False, engine='python')
    return resCSV

# Data Cleaning and Preparation Steps:
# Step 1: Replace missing job titles ('工作名称') with position names ('职位名称').
# Step 2: Remove entries where the publication date ('发布日期') is missing.
# Step 3: Exclude part-time jobs ('兼职') from the dataset.
# Step 4: Eliminate duplicates within a month considering '公司ID', '工作名称', '城市名称' as identifying fields.
# Step 5: Retain job postings from major websites only, based on the '数据来源' field.

dataNameTmp = "E:/Data/job_posting/Raw_data/job_posting_%s.dat"

naSum = 0
dupSum = 0

for i in range(1,142): 
    # Process each job posting file: Clean, filter, and save the processed data.

    curFile = dataNameTmp%i
        
    print(curFile," is Grouping Computing...")
    datDf = read_dat_data(curFile)
    datDf = datDf.replace(r'\N',np.NaN).dropna(subset=['发布日期'])
    datDf['工作名称'] = datDf[['工作名称']].replace(r'\N',np.NaN)
    datDf.loc[datDf['工作名称'].isna(),'工作名称'] = datDf.loc[datDf['工作名称'].isna(),'职位名称']
    datDf = datDf[datDf['工作名称'] != "兼职"]
     # subset the data to only include the '来源' == '智联招聘', '前程无忧', '拉勾网', 'BOSS直聘', '58同城', '猎聘网', '看准网', 百姓网', '拉勾网', '猎聘', '赶集网'， 'BOSS'
    datDf = datDf[datDf['数据来源'].isin(['智联招聘', '前程无忧', '拉勾网', 'BOSS直聘', '58同城', '猎聘网', '看准网', '百姓网', '拉勾网', '猎聘', '赶集网', 'BOSS'])]

    curNa = datDf.shape[0]
    naSum = naSum + curNa
    
    datDf['date'] = datDf['发布日期'].apply(lambda x: x[0:7])
    datDf = datDf.drop_duplicates(subset=['公司ID', '工作名称', '城市名称', 'date'], keep='first').reset_index(drop=True)
    
    curDup = datDf.shape[0]
    dupSum = dupSum + curDup
    
    print(curFile,"删除空值剩余: %s"%curNa, "去重复值剩余：%s"%curDup)
    datDf.to_csv('E:/Data/job_posting/mapped_job_posting/Update file/job_res_{}.csv'.format(i), sep='?', encoding = 'utf_8_sig', index=False)

# From the master data, we separate ``job description" so that rest of the data are manageable. 

directory = 'E:/Data/job_posting/mapped_job_posting/Update file/'

# iterate over files in that directory
for filename in os.listdir(directory):
    # checking if it is a file
    if filename.startswith("job_res_"): # for files start with a prefix #
        f = os.path.join(directory, filename)
        df = pd.read_csv(f, encoding = "utf_8_sig", on_bad_lines='skip', delimiter= "?", header=None, encoding_errors='ignore')
        df.rename(columns={0: '招聘主键ID', 1: '公司ID', 2: '公司名称', 3: '城市名称', 4: '公司所在区域', 5: '工作薪酬', 6: '教育要求', 
                   7: '工作经历', 8: '工作描述', 9: '职位名称', 10: '工作名称', 11: '招聘数量', 12: '发布日期', 13: '行业名称', 
                   14: '数据来源'}, inplace=True)
        df_charac = df[['招聘主键ID', '公司ID', '公司名称', '城市名称', '公司所在区域', '工作薪酬', '教育要求', '工作经历', '职位名称', 
         '工作名称', '招聘数量', '发布日期', '行业名称', '数据来源']]
        
        # export the data to csv, use the header and set the encoding to utf-8
        df_charac.to_csv('E:/Data/job_posting/processed/charac/{}'.format(filename), encoding = "utf_8_sig", header=True)

# Append all the character data together, then generate a list of the job titles that used to feed to the ChatGPT
os.chdir("E:/Data/job_posting/processed/charac")
extension = 'csv'
all_filenames = [i for i in glob.glob('*.{}'.format(extension))]
#combine all files in the list
combined_csv = pd.concat([pd.read_csv(f, encoding = "utf_8_sig", on_bad_lines='skip', usecols = ['工作名称']) for f in all_filenames], ignore_index=True)
# This is the complete list of job posting titles 
combined_csv.to_csv('E:/Data/job_posting/processed/estimation/charac_posting.csv', index=False, header=True)
# Save all the ``job description" data

directory = 'E:/Data/job_posting/mapped_job_posting/Update file/'
# iterate over files in that directory
for filename in os.listdir(directory):
    # checking if it is a file
    if filename.startswith("job_res_"): # for files start with a prefix #
        f = os.path.join(directory, filename)
        df = pd.read_csv(f, encoding = "utf_8_sig", on_bad_lines='skip', delimiter= "?", header=None, encoding_errors='ignore')
        df.rename(columns={0: '招聘主键ID', 1: '公司ID', 2: '公司名称', 3: '城市名称', 4: '公司所在区域', 5: '工作薪酬', 6: '教育要求', 
                   7: '工作经历', 8: '工作描述', 9: '职位名称', 10: '工作名称', 11: '招聘数量', 12: '发布日期', 13: '行业名称', 
                   14: '数据来源'}, inplace=True)
        df_desp = df[['招聘主键ID', '公司ID', '工作描述']]
        df_desp.to_csv('E:/Data/job_posting/processed/description/{}'.format(filename))
        
        del df_desp
        del df
        gc.collect()

### Label and form the representative data: `title_classification`
- `est_sample` is generated: the sample used for model fine-tune

In [ ]:
# Standard library imports
import gc
import json
import os
import re
import time
import urllib3

# Third-party library imports
import numpy as np
import openai
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from glob import glob
from sklearn.model_selection import train_test_split


# Set up the OpenAI organization and API key for subsequent API calls.
openai.organization = "org-**********************"
openai.api_key = "sk-**********************"
api_key = "sk-**********************"



### Load the job posting data, specifically extracting the '工作名称' (job titles) column. Determine the shape of the loaded DataFrame.
df = pd.read_csv('E:/Data/job_posting/processed/estimation/charac_posting.csv', encoding = "utf_8_sig", on_bad_lines='skip', usecols = ['工作名称'])




### Then, we clean this dataframe by filtering out the job posting titles with duplicates less than 5 times.
# Count the occurrences of each unique job title in the DataFrame to identify duplicates.
df_counted = df['工作名称'].value_counts()
# Filter out job titles that occur fewer than 5 times. The result is a filtered DataFrame with more frequently occurring titles.
df_filtered = df_counted[df_counted>5]
# Calculate the total number of job postings in the filtered DataFrame.
df_filtered.sum()
# Convert the series 'df_filtered' to a DataFrame and reset the index. Rename columns to '工作名称' for job titles and 'count' for their occurrences.
df_filtered = df_filtered.to_frame().reset_index()
df_filtered.columns = ['工作名称', 'count']




### Parallelize the job title classification using Python's ThreadPoolExecutor, we feed the job titles to ChatGPT to map it to a SOC category.
# Split the DataFrame into 300 smaller sub-DataFrames for more manageable processing.
sub_dfs = np.array_split(df_filtered, 300)

# Export each sub-DataFrame to a separate CSV file for further processing.
for i, sub_df in enumerate(sub_dfs):
    sub_df.to_csv(f'E:/Data/job_posting/processed/title_raw/title_{i+1}.csv', index=True, encoding = "utf_8_sig")
def classify_job_title(job_title, api_key):
    # Define a function to classify a given job title using the OpenAI API. The function sends a request to the OpenAI API and returns the SOC (Standard Occupational Classification) code for the given job title.
    url = 'https://api.openai.com/v1/chat/completions'
    headers = {'Content-Type': 'application/json',
               'Authorization': f'Bearer {api_key}'}
    data = {'model': 'gpt-3.5-turbo-0301',
            'messages':[
                {
                'role': 'user', 
                'content': f'The most likely Standard Occupational Classification title and code the occupation fall into: "{job_title}", only show the SOC code.'
                }
                       ]
            }
    try:
        response = requests.post(url, headers=headers, json=data, verify=False)
        response_data = json.loads(response.text)
        if response.status_code != 200:
            raise Exception(response_data['error']['message'])
        return response_data['choices'][0]['message']['content']
    except Exception as e:
        print(f'Error occurred: {e}')
        return 'N/A'








# Parallelize the job title classification using Python's ThreadPoolExecutor

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ["http_proxy"] = "http://127.0.0.1:10809"
os.environ["https_proxy"] = "http://127.0.0.1:10809"

# Set up a ThreadPoolExecutor with 30 worker threads for parallel processing.
with ThreadPoolExecutor(max_workers=30) as executor:
    # Loop through each file in the range and process job titles.
    for i in range(1, 301):
        # read the file into a dataframe
        filename = f'E:/Data/job_posting/processed/title_raw/title_{i}.csv'
        df = pd.read_csv(filename, encoding="utf_8_sig", on_bad_lines='skip')

        # Submit each job title in the file to the classify_job_title function using the thread pool.
        job_titles = df['工作名称'].tolist()
        futures = [executor.submit(classify_job_title, job_title, api_key) for job_title in job_titles]

        # wait for all threads to complete and get the results
        # create an empty list to store soc codes
        soc_codes = []
        for future in futures:
            soc_code = future.result()
            soc_codes.append(soc_code)

        # Append the SOC codes obtained from the classification to the DataFrame.
        df['soc_code'] = soc_codes
        # Extract only the SOC code part from the results.
        df['soc_code'] = df['soc_code'].str.extract(r'(\d{2}-\d{4})')
        # Save the updated DataFrame to a new CSV file.
        df.to_csv(f'E:/Data/job_posting/processed/title_mapped/title_{i}.csv', encoding="utf_8_sig") 
        del df


 

### Combine all processed CSV files into a single DataFrame.
path = r'F:/Data/job_posting/processed/title_mapped' # use your path
all_files = glob(os.path.join(path, "*.csv"))
df_from_each_file = (pd.read_csv(f, encoding="utf_8_sig") for f in all_files)
df_title = pd.concat(df_from_each_file, ignore_index=True)
# Keep only the '工作名称' and 'soc_code' columns and drop rows with missing SOC codes.
df_title = df_title[['工作名称', 'soc_code']].dropna(subset=['soc_code'])



### The next step is to map the rest unmapped job postings to the SOCs using the job descriptions.
directory = 'F:/Data/job_posting/mapped_job_posting/Update file/'
# create an empty DataFrame to store merged data
df_titleLabel = pd.DataFrame()

# Iterate over files in the directory and merge them with the 'df_title' DataFrame.
for filename in os.listdir(directory):
    if filename.startswith("job_res_"):
        f = os.path.join(directory, filename)
        df = pd.read_csv(f, encoding="utf_8_sig", on_bad_lines='skip', delimiter="?")
        df.rename(columns={'招聘ID': '招聘主键ID'}, inplace=True)
        df = df[['招聘主键ID', '工作描述', '工作名称']]
        df_titleData = pd.merge(df, df_title, on='工作名称', how='inner')
        df_titleLabel = df_titleLabel.append(df_titleData)

    # Convert the 'soc_code' column to a string type for consistency.
    df_titleLabel['soc_code'] = df_titleLabel['soc_code'].astype(str)



### To do this, we first load the dataframe from ONET which contains all the possible SOC job titles. This step helps to remove incorrect mapping.
df_soc = pd.read_csv('F:/Data/job_posting/processed/2019_to_SOC_Crosswalk.csv')
df_soc = df_soc[['2018 SOC Code']].drop_duplicates()
df_soc['2018 SOC Code'] = df_soc['2018 SOC Code'].str[:-1] + '0'
df_soc.rename(columns={'2018 SOC Code': 'soc_code'}, inplace=True)
df_soc = df_soc.drop_duplicates(subset=['soc_code'], keep='first')


### We set the finest level to 6 digits, which already covers 459 broad occupations. The concern of going into more granular level is that the number of job postings will be too small to train a good model.
# However, not all the broad occupations show up equally in the dataset. We filter out the broad occupations with less than 100 job postings. In this way, it has enough observation to train a good model.
# We end up with 408 broad occupations. We randonmly sample 3000 job postings within each broad occupations, and save them to csv files. We feed this data to ChatGPT to map the job postings to the SOC categories by using job descriptions.

# replace the last digit of 'soc_code' with '0'
df_titleLabel['soc_code'] = df_titleLabel['soc_code'].str[:-1] + '0'
# merge the 'test_dfSoc' and 'df_soc' using 'soc_code', only keep the matched sample
df_titlelabel = pd.merge(df_titleLabel, df_soc, on='soc_code', how='inner')
# keep number of observations by 'soc_code' is more than 100
df_titlelabel = df_titlelabel.groupby('soc_code').filter(lambda x: len(x) > 100)
# save the final merged DataFrame to a csv file
df_titlelabel.to_csv('F:/Data/job_posting/processed/finetune/df_titleLabel.csv', index=False, encoding = "utf_8_sig", header=True, quoting=csv.QUOTE_NONNUMERIC)
# randomly select 3000 samples within each unique value of 'soc_code'
df = df_titlelabel.groupby('soc_code', group_keys=False).apply(lambda x: x.sample(min(len(x), 3000)))
df.to_csv('F:/Data/job_posting/processed/finetune/secondcheck_sample.csv', index=False, encoding = "utf_8_sig", header=True, quoting=csv.QUOTE_NONNUMERIC)




### Afer first labelling based on the title of job posting, we again use GPT to verify the labelling based on the description of job posting.
def classify_job_desp(desp, job_title, api_key):
    # The function sends a request to OpenAI's API and returns a yes/no response on the classification accuracy.
    url = 'https://api.openai.com/v1/chat/completions'
    headers = {'Content-Type': 'application/json',
               'Authorization': f'Bearer {api_key}'}
    data = {'model': 'gpt-3.5-turbo-0301',
            'messages':[
                {
                'role': 'user', 
                'content': f"Based on this job description (in Chinese): '{desp}' Is this Standard Occupational Classification code: '{job_title}' a reasonable classification (at broad group level)? Only tell me yes or no."
                }
                       ]
            }
    try:
        response = requests.post(url, headers=headers, json=data, verify=False)
        response_data = json.loads(response.text)
        if response.status_code != 200:
            raise Exception(response_data['error']['message'])
        return response_data['choices'][0]['message']['content']
    except Exception as e:
        print(f'Error occurred: {e}')
        return 'N/A'

### Double check on the sub-sampled dataset
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) # Disable SSL warnings #

# Set up parallel processing for validating job descriptions.
df = pd.read_csv('F:/Data/job_posting/processed/finetune/secondcheck_sample.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

# Create a list of tuples containing (desp, job_title) pairs
desps = df['工作描述'].tolist()
job_titles = df['soc_code'].tolist()
desp_job_title_pairs = list(zip(desps, job_titles))

# Function to handle ThreadPoolExecutor map
def classify_wrapper(args):
    return classify_job_desp(*args)

# Create a thread pool with 30 worker threads
with ThreadPoolExecutor(max_workers=20) as executor:
    # Submit desp_job_title_pairs to the thread pool
    true_inds = list(executor.map(classify_wrapper, [(desp, job_title, api_key) for desp, job_title in desp_job_title_pairs]))

# Append the validation results to the DataFrame and clean the data.
df['true_ind'] = true_inds
# Clean the soc_codes: remove '.' from 'true_ind'
df['true_ind'] = df['true_ind'].str.replace('.', '')
df.to_csv('F:/Data/job_posting/processed/finetune/secondcheck_sample_ind.csv', encoding="utf_8_sig")






### Generate the final dataset for model finetune

#  The first screen is filtered based on the job title, and we left with 32 million-ish job postings.
#  The second screen is based on the job description. From the 32 million-ish job postings, we randomly sample 3000 observations within each unique SOC category and feed them to GPT for the second check.
#  We keep the samples that pass the second check, which means the SOC category predicted by GPT is the same as the SOC category predicted by the job title. This is the first part of the final dataset.
#  The second part of the final dataset is the job postings that are first screened by the job title, but has not been selected by the random sampling for the second screen. Within this data pool, we randomly sample 5000 observations within each unique SOC category. This is the second part of the final dataset.
#  We combine the two parts of the final dataset, and we assign weight 1 for the second check sample and assign weight 0.5 for the random sample. This is the final dataset for model finetune.

df_sample = pd.read_csv('F:/Data/job_posting/processed/finetune/secondcheck_sample_ind.csv', encoding="utf_8_sig")

# keep if true_ind is 'Yes'
df_sample = df_sample[df_sample['true_ind'] == 'Yes']

# Read another dataset 'df_titleLabel' from the specified CSV file.
df_titlelabel = pd.read_csv('F:/Data/job_posting/processed/finetune/df_titleLabel.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

# Merge 'df_sample' and 'df_titlelabel' on '招聘主键ID' using an outer join, and add a merge indicator column.
merged_df = df_sample.merge(df_titlelabel, on='招聘主键ID', how='outer', indicator=True)

# Select rows from 'merged_df' that are only in 'df_titlelabel' (right_only) and specific columns.
filtered_df = merged_df.loc[merged_df['_merge'] == 'right_only', ['招聘主键ID', '工作描述_y', '工作名称_y', 'soc_code_y']]

# Rename columns in 'filtered_df' by removing the '_y' suffix added during the merge.
filtered_df = filtered_df.rename(columns={'工作描述_y': '工作描述', '工作名称_y': '工作名称', 'soc_code_y': 'soc_code'})

# Drop rows from 'filtered_df' where the '工作描述' (job description) column has null values.
filtered_df = filtered_df.dropna(subset=['工作描述'])

# Within each 'soc_code' group, randomly select up to 5000 samples from 'filtered_df'.
filtered_df = filtered_df.groupby('soc_code', group_keys=False).apply(lambda x: x.sample(min(len(x), 5000)))

# append 'df_sample' and 'filtered_df', create a new dataframe 'df'
df = pd.concat([df_sample, filtered_df], axis = 0)

# Keep rows in 'df' where the 'soc_code' does not start with '55'.
df = df[~df['soc_code'].str.startswith('55')]
df.to_csv('F:/Data/job_posting/processed/finetune/est_sample.csv', index=False, encoding = "utf_8_sig", header=True, quoting=csv.QUOTE_NONNUMERIC)


### Fine-tune the Chinese BERT_wwm model: `flat_classification`
- This code is run on HPC with 6 GPUs. The output is the fine-tuned model and classification results.

In [ ]:
# Standard library imports
import csv
import os

# Third-party library imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from transformers import (AdamW, BertForSequenceClassification, BertTokenizer, get_linear_schedule_with_warmup, get_scheduler, Trainer)


tokenizer = BertTokenizer.from_pretrained("D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/chinese-bert-wwm/")

df = pd.read_csv("G:/Data/job_posting/processed/finetune/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
df['soc_code'] = df['soc_code'].str.replace('-', '')
# replace 'Yes' with True and NaN with False using the fillna() and astype() methods
df['true_ind'] = df['true_ind'].fillna(False).astype(bool)
# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order. Create a dictionary to map unique soc_codes to sequential integer labels
unique_soc_codes = sorted(df['soc_code'].unique())
soc_code_dict  = {soc_code: i for i, soc_code in enumerate(unique_soc_codes)}





### The dataset is split into train, validation, and test sets as follows:
# The initial train_test_split call splits df into train_df_sample (60% of the data) and temp_df_sample (40% of the data).
# The second train_test_split call further splits temp_df_sample into valid_df_sample (50% of temp_df_sample, or 20% of the original data) and test_df_sample (50% of temp_df_sample, or 20% of the original data).
# So, the final ratio of the dataset split is 60% for training, 20% for validation, and 20% for testing. Create into train, validation and test set

train_df_sample, temp_df_sample = train_test_split(df, test_size=0.4, random_state=42)
valid_df_sample, test_df_sample = train_test_split(temp_df_sample, test_size=0.5, random_state=42)

# export the train, validation and test set to csv
train_df_sample.to_csv('F:/Data/job_posting/processed/finetune/train_df_sample.csv', index=False, encoding = "utf_8_sig", header=True)
test_df_sample.to_csv('F:/Data/job_posting/processed/finetune/test_df_sample.csv', index=False, encoding = "utf_8_sig", header=True)
valid_df_sample.to_csv('F:/Data/job_posting/processed/finetune/valid_df_sample.csv', index=False, encoding = "utf_8_sig", header=True)

def preprocess_df(df, soc_code_dict):
    # Drop the 'Unnamed: 0' column
    df = df.drop(['Unnamed: 0'], axis=1)
    # Generate a new column 'soc_code1' with mapped values from 'soc_code'
    df['soc_code1'] = df['soc_code'].map(soc_code_dict)
    return df

# Applying the function to each DataFrame
train_df_sample = preprocess_df(train_df_sample, soc_code_dict)
test_df_sample = preprocess_df(test_df_sample, soc_code_dict)
valid_df_sample = preprocess_df(valid_df_sample, soc_code_dict)

# Function to extract titles, texts, and labels from a DataFrame
def extract_data(df):
    titles = df['工作名称'].astype(str).tolist()
    texts = df['工作描述'].astype(str).tolist()
    labels = df['soc_code1'].tolist()
    return titles, texts, labels

# Extracting data from each DataFrame
train_titles, train_texts, train_labels = extract_data(train_df_sample)
test_titles, test_texts, test_labels = extract_data(test_df_sample)
valid_titles, valid_texts, valid_labels = extract_data(valid_df_sample)

# If you have more "credible" or reliable labels in your dataset, you can leverage this information to improve the performance of your model by assigning different weights to the loss function during training. This way, the model will put more emphasis on learning from the credible samples.
def assign_weight(true_ind):
    if true_ind:
        return 1.0
    else:
        return 0.5
    
# True for credible labels and False for less credible labels
train_df_sample['weight'] = train_df_sample['true_ind'].apply(assign_weight)
test_df_sample['weight'] = test_df_sample['true_ind'].apply(assign_weight)
valid_df_sample['weight'] = valid_df_sample['true_ind'].apply(assign_weight)

# Extract the weights
train_weights = train_df_sample['weight'].tolist()
valid_weights = valid_df_sample['weight'].tolist()
test_weights = test_df_sample['weight'].tolist()

# Create the JobPostingDataset class
class JobPostingDataset(Dataset):
    def __init__(self, titles, descriptions, labels, weights, tokenizer, max_length):
        self.titles = titles
        self.descriptions = descriptions
        self.labels = labels
        self.weights = weights
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title = self.titles[idx]
        description = self.descriptions[idx]
        label = self.labels[idx]
        weight = self.weights[idx]

        # Concatenate title and description, repeat the title to give it more importance
        repeat_title = 2  # Adjust this value to control the importance of the title
        text = (title + " ") * repeat_title + description

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Return a tuple of the input tensors, label, and weight
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(weight, dtype=torch.float),
        )

# Create the datasets
max_length = 512
train_dataset = JobPostingDataset(train_titles, train_texts, train_labels, train_weights, tokenizer, max_length)
valid_dataset = JobPostingDataset(valid_titles, valid_texts, valid_labels, valid_weights, tokenizer, max_length)
test_dataset = JobPostingDataset(test_titles, test_texts, test_labels, test_weights, tokenizer, max_length)

# Create the data loaders
batch_size = 20
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
### Define an evaluate() function to compute the validation loss
def evaluate(model, valid_loader, device, loss_fn):
    model.eval()
    total_loss = 0
    num_batches = 0
    with torch.no_grad():
        for batch in valid_loader:
            inputs = batch[0].to(device)
            masks = batch[1].to(device)
            labels = batch[2].to(device)
            weights = batch[3].to(device)

            logits = model(inputs, attention_mask=masks).logits
            batch_loss = loss_fn(logits, labels)
            weighted_batch_loss = batch_loss * weights
            loss = torch.mean(weighted_batch_loss)

            total_loss += loss.item()
            num_batches += 1
    return total_loss / num_batches







# Handle class imbalance by adjusting class weights in the CrossEntropyLoss criterion. To achieve this, you need to compute class weights and pass them as an argument to the CrossEntropyLoss function.
# The training loop to include early stopping based on the validation loss.
# Define the model, the optimizer, and the learning rate scheduler

num_labels = len(train_df_sample['soc_code1'].unique())
model = BertForSequenceClassification.from_pretrained("G:/Other computers/我的计算机/cloud_share/Job_posting_data/chinese-bert-wwm/", num_labels=num_labels)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
num_epochs = 300
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)



# Compute class weights using the train_labels_np array
unique_labels = train_df_sample['soc_code1'].unique()
class_weights = compute_class_weight('balanced', classes=unique_labels, y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float) 

# The patience parameter determines how many consecutive epochs the model can go without an improvement in validation loss before stopping the training. 
# In this case, the patience is set to 3, meaning that if the validation loss does not improve for 3 consecutive epochs, the training will be stopped.
early_stopping_patience = 3

# This line initializes a counter variable called num_epochs_without_improvement that keeps track of the number of consecutive epochs without an improvement in validation loss. 
# The counter is set to 0 at the beginning of the training process and is incremented by 1 whenever there is no improvement in the validation loss. If the validation loss improves in a particular epoch, the counter is reset to 0.
num_epochs_without_improvement = 0

# During the training loop, if num_epochs_without_improvement becomes equal to or greater than early_stopping_patience, the training will be stopped. 
# This way, the training process can be terminated early when the model starts overfitting, or when there is no significant improvement in the validation loss.
best_valid_loss = float('inf')

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize lists to store losses
train_losses = []

# Utilize multiple GPUs with DataParallel
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

# Pass the computed class_weights to the CrossEntropyLoss function:
# Create a loss function that doesn't reduce the losses right away and pass class_weights
# By incorporating class weights into the loss function, the model will pay more attention to the minority classes during training. 
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device), reduction='none')

model.train()
for epoch in range(num_epochs):
    epoch_train_loss = 0
    num_batches = 0
    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)
        weights = batch[3].to(device)  # Assuming the weights are the 4th element in the batch

        optimizer.zero_grad()

        logits = model(inputs, attention_mask=masks).logits

        # Compute the loss for each sample
        batch_loss = loss_fn(logits, labels)

        # Multiply the loss by the corresponding weight
        weighted_batch_loss = batch_loss * weights

        # Average the weighted losses
        loss = torch.mean(weighted_batch_loss)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # Add the current batch loss to the epoch_train_loss
        epoch_train_loss += loss.item()
        num_batches += 1

    # Calculate average loss for the current epoch and append it to the train_losses list
    epoch_train_loss /= num_batches
    train_losses.append(epoch_train_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_train_loss:.4f}")

    # Evaluate the model on the validation set
    valid_loss = evaluate(model, valid_loader, device, loss_fn)
    print(f"Validation Loss: {valid_loss:.4f}")

    # Save the best model based on the validation loss
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        if isinstance(model, torch.nn.DataParallel):
            model.module.save_pretrained("F:/Data/job_posting/processed/finetune/best_model")
        else:
            model.save_pretrained("F:/Data/job_posting/processed/finetune/best_model")
        num_epochs_without_improvement = 0
    else:
        num_epochs_without_improvement += 1

    # Check the stopping condition and break the loop if needed
    if num_epochs_without_improvement >= early_stopping_patience:
        print("Early stopping due to no improvement in validation loss.")
        break

#### `bert_robust_eva`

In [ ]:
from transformers import BertTokenizer
import torch
import numpy as np
import pandas as pd
import csv
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight





batch_size = 50
tokenizer = BertTokenizer.from_pretrained("/share/home/320346/chinese-bert-wwm/")
df = pd.read_csv("/share/home/320346/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
df['soc_code'] = df['soc_code'].str.replace('-', '')
# replace 'Yes' with True and NaN with False using the fillna() and astype() methods
df['true_ind'] = df['true_ind'].fillna(False).astype(bool)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
# Create a dictionary to map unique soc_codes to sequential integer labels
unique_soc_codes = sorted(df['soc_code'].unique())
soc_code_dict  = {soc_code: i for i, soc_code in enumerate(unique_soc_codes)}








train_df_sample, temp_df_sample = train_test_split(df, test_size=0.4, random_state=42)
valid_df_sample, test_df_sample = train_test_split(temp_df_sample, test_size=0.5, random_state=42)

train_df_sample.to_csv("/share/home/320346/train_df_sample.csv", index=False, encoding = "utf_8_sig", header=True)
test_df_sample.to_csv("/share/home/320346/test_df_sample.csv", index=False, encoding = "utf_8_sig", header=True)
valid_df_sample.to_csv("/share/home/320346/valid_df_sample.csv", index=False, encoding = "utf_8_sig", header=True)

# drop index column, 'true_ind' and 'sample' columns
train_df_sample = train_df_sample.drop(['Unnamed: 0'], axis = 1)
# keep if 'true_ind' is False
train_df_sample = train_df_sample[train_df_sample['true_ind'] == True]
# Generate a new column 'soc_code1' with the mapped values from 'soc_code'
train_df_sample['soc_code1'] = train_df_sample['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
test_df_sample = test_df_sample.drop(['Unnamed: 0'], axis = 1)
# keep if 'true_ind' is False
test_df_sample = test_df_sample[test_df_sample['true_ind'] == True]
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the test set
test_df_sample['soc_code1'] = test_df_sample['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
valid_df_sample = valid_df_sample.drop(['Unnamed: 0'], axis = 1)
# keep if 'true_ind' is False
valid_df_sample = valid_df_sample[valid_df_sample['true_ind'] == True]
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the validation set
valid_df_sample['soc_code1'] = valid_df_sample['soc_code'].map(soc_code_dict)

# Tokenize the text and convert it into input features
train_titles = train_df_sample['工作名称'].astype(str).tolist()
train_texts = train_df_sample['工作描述'].astype(str).tolist()
train_labels = train_df_sample['soc_code1'].tolist()

test_titles = test_df_sample['工作名称'].astype(str).tolist()
test_texts = test_df_sample['工作描述'].astype(str).tolist()
test_labels = test_df_sample['soc_code1'].tolist()

valid_titles = valid_df_sample['工作名称'].astype(str).tolist()
valid_texts = valid_df_sample['工作描述'].astype(str).tolist()
valid_labels = valid_df_sample['soc_code1'].tolist()

def assign_weight(true_ind):
    if true_ind:
        return 1.0
    else:
        return 0.5
    
# Assuming you have a column called 'true_ind' in your dataset with boolean values
# True for credible labels and False for less credible labels
train_df_sample['weight'] = train_df_sample['true_ind'].apply(assign_weight)
test_df_sample['weight'] = test_df_sample['true_ind'].apply(assign_weight)
valid_df_sample['weight'] = valid_df_sample['true_ind'].apply(assign_weight)

# Extract the weights
train_weights = train_df_sample['weight'].tolist()
valid_weights = valid_df_sample['weight'].tolist()
test_weights = test_df_sample['weight'].tolist()

# Create the JobPostingDataset class
class JobPostingDataset(Dataset):
    def __init__(self, titles, descriptions, labels, weights, tokenizer, max_length):
        self.titles = titles
        self.descriptions = descriptions
        self.labels = labels
        self.weights = weights
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title = self.titles[idx]
        description = self.descriptions[idx]
        label = self.labels[idx]
        weight = self.weights[idx]

        # Concatenate title and description, repeat the title to give it more importance
        repeat_title = 2  # Adjust this value to control the importance of the title
        text = (title + " ") * repeat_title + description

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Return a tuple of the input tensors, label, and weight
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(weight, dtype=torch.float),
        )











# Create the datasets
max_length = 512
train_dataset = JobPostingDataset(train_titles, train_texts, train_labels, train_weights, tokenizer, max_length)
valid_dataset = JobPostingDataset(valid_titles, valid_texts, valid_labels, valid_weights, tokenizer, max_length)
test_dataset = JobPostingDataset(test_titles, test_texts, test_labels, test_weights, tokenizer, max_length)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)








def evaluate(model, valid_loader, device, loss_fn):
    model.eval()
    total_loss = 0
    num_batches = 0
    with torch.no_grad():
        for batch in valid_loader:
            inputs = batch[0].to(device)
            masks = batch[1].to(device)
            labels = batch[2].to(device)
            weights = batch[3].to(device)

            logits = model(inputs, attention_mask=masks).logits
            batch_loss = loss_fn(logits, labels)
            weighted_batch_loss = batch_loss * weights
            loss = torch.mean(weighted_batch_loss)

            total_loss += loss.item()
            num_batches += 1
    return total_loss / num_batches










# Define the model, the optimizer, and the learning rate scheduler
num_labels = len(train_df_sample['soc_code1'].unique())

model = BertForSequenceClassification.from_pretrained("/share/home/320346/chinese-bert-wwm/", num_labels=num_labels)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
num_epochs = 30
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

unique_labels = train_df_sample['soc_code1'].unique()
class_weights = compute_class_weight('balanced', classes=unique_labels, y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)








early_stopping_patience = 3
# This line initializes a counter variable called num_epochs_without_improvement that keeps track of the number of consecutive epochs without an improvement in validation loss. 
# The counter is set to 0 at the beginning of the training process and is incremented by 1 whenever there is no improvement in the validation loss. If the validation loss improves in a particular epoch, the counter is reset to 0.
num_epochs_without_improvement = 0
# During the training loop, if num_epochs_without_improvement becomes equal to or greater than early_stopping_patience, the training will be stopped. 
# This way, the training process can be terminated early when the model starts overfitting, or when there is no significant improvement in the validation loss.
best_valid_loss = float('inf')

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Initialize lists to store losses
train_losses = []

# Utilize multiple GPUs with DataParallel
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

# Pass the computed class_weights to the CrossEntropyLoss function:
# Create a loss function that doesn't reduce the losses right away and pass class_weights
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device), reduction='none')
# By incorporating class weights into the loss function, the model will pay more attention to the minority classes during training. 

model.train()
for epoch in range(num_epochs):
    epoch_train_loss = 0
    num_batches = 0
    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)
        weights = batch[3].to(device)  # Assuming the weights are the 4th element in the batch

        optimizer.zero_grad()

        logits = model(inputs, attention_mask=masks).logits

        # Compute the loss for each sample
        batch_loss = loss_fn(logits, labels)

        # Multiply the loss by the corresponding weight
        weighted_batch_loss = batch_loss * weights

        # Average the weighted losses
        loss = torch.mean(weighted_batch_loss)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # Add the current batch loss to the epoch_train_loss
        epoch_train_loss += loss.item()
        num_batches += 1

    # Calculate average loss for the current epoch and append it to the train_losses list
    epoch_train_loss /= num_batches
    train_losses.append(epoch_train_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_train_loss:.4f}")

    # Evaluate the model on the validation set
    valid_loss = evaluate(model, valid_loader, device, loss_fn)
    print(f"Validation Loss: {valid_loss:.4f}")

    # Save the best model based on the validation loss
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        if isinstance(model, torch.nn.DataParallel):
            model.module.save_pretrained("/share/home/320346/bert_robust_model")
        else:
            model.save_pretrained("/share/home/320346/bert_robust_model")
        num_epochs_without_improvement = 0
    else:
        num_epochs_without_improvement += 1

    # Check the stopping condition and break the loop if needed
    if num_epochs_without_improvement >= early_stopping_patience:
        print("Early stopping due to no improvement in validation loss.")
        break







# Evaluate the model and store predictions
model.eval()
predictions = []
true_labels = []

# Create a reverse mapping dictionary to convert the sequential labels back to SOC labels
reverse_soc_code_dict = {v: k for k, v in soc_code_dict.items()}

with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)

        logits = model(inputs, attention_mask=masks).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels = labels.cpu().numpy()

        predictions.extend(preds)
        true_labels.extend(labels)

# Convert predictions and true_labels into NumPy arrays
predictions = np.array(predictions)
true_labels = np.array(true_labels)

# Map the sequential labels back to the original SOC codes
soc_predictions = [reverse_soc_code_dict[p] for p in predictions]
soc_true_labels = [reverse_soc_code_dict[l] for l in true_labels]

# Compute classification report
unique_labels = sorted(list(set(soc_true_labels).union(set(soc_predictions))))
report = classification_report(soc_true_labels, soc_predictions, labels=unique_labels, output_dict=True)

# Convert report to a pandas DataFrame
report_df = pd.DataFrame(report).transpose()
report_df.to_csv('/share/home/320346/bert_report_df.csv', index=True)

#### `cnn_eva`

In [ ]:
# Standard library imports
import itertools

# Third-party library imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from transformers import (BertForSequenceClassification, BertTokenizer)




tokenizer = BertTokenizer.from_pretrained("/share/home/320346/chinese-bert-wwm/")
batch_size = 200

df = pd.read_csv("/share/home/320346/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
df['soc_code'] = df['soc_code'].str.replace('-', '')
# replace 'Yes' with True and NaN with False using the fillna() and astype() methods
df['true_ind'] = df['true_ind'].fillna(False).astype(bool)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
# Create a dictionary to map unique soc_codes to sequential integer labels
unique_soc_codes = sorted(df['soc_code'].unique())
soc_code_dict  = {soc_code: i for i, soc_code in enumerate(unique_soc_codes)}
# Load datasets from CSV files
train_df_sample = pd.read_csv('/share/home/320346/train_df_sample.csv', encoding="utf_8_sig")
valid_df_sample = pd.read_csv('/share/home/320346/valid_df_sample.csv', encoding="utf_8_sig")
test_df_sample = pd.read_csv('/share/home/320346/test_df_sample.csv', encoding="utf_8_sig")

# drop index column, 'true_ind' and 'sample' columns
train_df_sample = train_df_sample.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code'
train_df_sample['soc_code'] = train_df_sample['soc_code'].astype(str)
train_df_sample['soc_code1'] = train_df_sample['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
test_df_sample = test_df_sample.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the test set
test_df_sample['soc_code'] = test_df_sample['soc_code'].astype(str)
test_df_sample['soc_code1'] = test_df_sample['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
valid_df_sample = valid_df_sample.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the validation set
valid_df_sample['soc_code'] = valid_df_sample['soc_code'].astype(str)
valid_df_sample['soc_code1'] = valid_df_sample['soc_code'].map(soc_code_dict)




# Tokenize the text and convert it into input features
train_titles = train_df_sample['工作名称'].astype(str).tolist()
train_texts = train_df_sample['工作描述'].astype(str).tolist()
train_labels = train_df_sample['soc_code1'].tolist()


test_titles = test_df_sample['工作名称'].astype(str).tolist()
test_texts = test_df_sample['工作描述'].astype(str).tolist()
test_labels = test_df_sample['soc_code1'].tolist()


valid_titles = valid_df_sample['工作名称'].astype(str).tolist()
valid_texts = valid_df_sample['工作描述'].astype(str).tolist()
valid_labels = valid_df_sample['soc_code1'].tolist()










def assign_weight(true_ind):
    if true_ind:
        return 1.0
    else:
        return 0.5
    
# Assuming you have a column called 'true_ind' in your dataset with boolean values
# True for credible labels and False for less credible labels
train_df_sample['weight'] = train_df_sample['true_ind'].apply(assign_weight)
test_df_sample['weight'] = test_df_sample['true_ind'].apply(assign_weight)
valid_df_sample['weight'] = valid_df_sample['true_ind'].apply(assign_weight)

# Extract the weights
train_weights = train_df_sample['weight'].tolist()
valid_weights = valid_df_sample['weight'].tolist()
test_weights = test_df_sample['weight'].tolist()









# Create the JobPostingDataset class
class JobPostingDataset(Dataset):
    def __init__(self, titles, descriptions, labels, weights, tokenizer, max_length):
        self.titles = titles
        self.descriptions = descriptions
        self.labels = labels
        self.weights = weights
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title = self.titles[idx]
        description = self.descriptions[idx]
        label = self.labels[idx]
        weight = self.weights[idx]

        # Concatenate title and description, repeat the title to give it more importance
        repeat_title = 2  # Adjust this value to control the importance of the title
        text = (title + " ") * repeat_title + description

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Return a tuple of the input tensors, label, and weight
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(weight, dtype=torch.float),
        )











# Create the datasets
max_length = 512
train_dataset = JobPostingDataset(train_titles, train_texts, train_labels, train_weights, tokenizer, max_length)
valid_dataset = JobPostingDataset(valid_titles, valid_texts, valid_labels, valid_weights, tokenizer, max_length)
test_dataset = JobPostingDataset(test_titles, test_texts, test_labels, test_weights, tokenizer, max_length)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=20, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)







# Calculate Class Weights
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Assuming 'train_labels' is a list of your training labels
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# Modify Loss Function
# Define the weighted loss function
criterion = nn.CrossEntropyLoss(weight=class_weights)




class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # 1D Convolutional Layer
        self.conv1d = nn.Conv1d(in_channels=embed_dim, out_channels=32, kernel_size=3)

        # Global Max Pooling Layer
        self.global_max_pool = nn.AdaptiveMaxPool1d(1)

        # Intermediate Dense Layer with 250 hidden units
        self.fc1 = nn.Linear(32, 250)

        # Output Layer with softmax activation for classification
        self.fc2 = nn.Linear(250, num_classes)

    def forward(self, x):
        # Embedding layer
        x = self.embedding(x).permute(0, 2, 1)  # Rearrange to [batch, channels, sequence length]

        # Convolution and ReLU activation
        x = F.relu(self.conv1d(x))

        # Global Max Pooling
        x = self.global_max_pool(x).squeeze(2)

        # Intermediate Dense Layer
        x = F.relu(self.fc1(x))

        # Output Layer
        x = self.fc2(x)

        # Softmax activation
        return F.log_softmax(x, dim=1)

# Hyperparameters for grid search
hidden_dims = [25, 50, 100]
dropouts = [0.1, 0.2, 0.5]  # Assuming you're adding dropout to your model

# Other fixed hyperparameters
vocab_size = len(tokenizer)  # Replace with your actual vocab size
embed_dim = 100
num_classes = 406  # Replace with your actual number of classes
epochs = 10

# Grid search
best_hyperparams = {'hidden_dim': None, 'dropout': None}
best_valid_loss = float('inf')

for hidden_dim, dropout in itertools.product(hidden_dims, dropouts):
    print(f"Training with hidden_dim: {hidden_dim}, dropout: {dropout}")

    # Initialize the model with current hyperparameters
    model = TextCNN(vocab_size, embed_dim, num_classes)

    # Check if multiple GPUs are available and wrap the model using nn.DataParallel
    if torch.cuda.device_count() > 1:
        print("Using", torch.cuda.device_count(), "GPUs!")
        model = nn.DataParallel(model)

    model.to(device)

    # Define the optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    # Use the weighted loss function
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Training and Validation loop
    for epoch in range(epochs):
        train_loss = 0.0
        model.train()

        for inputs, _, labels, weights in train_loader:  # Ensure weights are returned by the loader
            inputs, labels, weights = inputs.to(device), labels.to(device), weights.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)

            # Calculate loss without reduction
            loss = F.cross_entropy(outputs, labels, reduction='none')
            # Apply sample weights
            weighted_loss = (loss * weights).mean()  # Average the weighted loss
            
            # Backward pass and optimize
            weighted_loss.backward()
            optimizer.step()
            
            train_loss += weighted_loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)

        # Validation phase
        valid_loss = 0.0
        model.eval()
        with torch.no_grad():
            for inputs, _, labels, _ in valid_loader:  # Ignore attention mask and weights
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                valid_loss += loss.item()
        valid_loss /= len(valid_loader)

        print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}')

    # Update best hyperparameters and save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_hyperparams['hidden_dim'] = hidden_dim
        best_hyperparams['dropout'] = dropout

        # Save the best model
        torch.save(model.state_dict(), '/share/home/320346/cnn_model_state_dict.pth')  # Update path as necessary

print(f"Best Hyperparameters: Hidden Dim - {best_hyperparams['hidden_dim']}, Dropout - {best_hyperparams['dropout']}")





model = TextCNN(vocab_size, embed_dim, num_classes)
# Load the saved state dict for the CNN model
cnn_state_dict = torch.load('/share/home/320346/cnn_model_state_dict.pth')

# Adjust for the keys if it was saved with nn.DataParallel
if list(cnn_state_dict.keys())[0].startswith('module.'):
    cnn_state_dict = {key[len("module."):]: value for key, value in cnn_state_dict.items()}

# Load the adjusted state dict into your CNN model
model.load_state_dict(cnn_state_dict)

# Create a reverse mapping dictionary to convert the sequential labels back to SOC labels
reverse_soc_code_dict = {v: k for k, v in soc_code_dict.items()}

# Evaluate the model and store predictions
model = model.to(device)
model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for inputs, _, labels, _ in test_loader:  # Ignore attention masks
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        labels = labels.cpu().numpy()

        predictions.extend(preds)
        true_labels.extend(labels)

# Debug: Print the lengths of predictions and true_labels
print(f"Length of predictions: {len(predictions)}")
print(f"Length of true_labels: {len(true_labels)}")

# Convert predictions and true_labels into NumPy arrays
predictions = np.array(predictions)
true_labels = np.array(true_labels)

# Assuming 'reverse_soc_code_dict' is a dictionary that maps the numeric labels back to the original SOC codes
soc_predictions = [reverse_soc_code_dict[p] for p in predictions]
soc_true_labels = [reverse_soc_code_dict[l] for l in true_labels]

# Compute classification report
unique_labels = sorted(list(set(soc_true_labels).union(set(soc_predictions))))
report = classification_report(soc_true_labels, soc_predictions, labels=unique_labels, output_dict=True)

# Convert report to a pandas DataFrame
report_df = pd.DataFrame(report).transpose()

# Print the report
print(report_df)

# Save the report to a CSV file
report_df.to_csv('/share/home/320346/cnn_report_df.csv', index=True)

#### `lstm_eva`

In [ ]:
# Standard library imports
import itertools

# Third-party library imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from transformers import (BertForSequenceClassification, BertTokenizer)



tokenizer = BertTokenizer.from_pretrained("/share/home/320346/chinese-bert-wwm/")
batch_size = 250

df = pd.read_csv("/share/home/320346/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')

# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
df['soc_code'] = df['soc_code'].str.replace('-', '')
# replace 'Yes' with True and NaN with False using the fillna() and astype() methods
df['true_ind'] = df['true_ind'].fillna(False).astype(bool)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
# Create a dictionary to map unique soc_codes to sequential integer labels
unique_soc_codes = sorted(df['soc_code'].unique())
soc_code_dict  = {soc_code: i for i, soc_code in enumerate(unique_soc_codes)}
# Load datasets from CSV files
train_df_sample = pd.read_csv('/share/home/320346/train_df_sample.csv', encoding="utf_8_sig")
valid_df_sample = pd.read_csv('/share/home/320346/valid_df_sample.csv', encoding="utf_8_sig")
test_df_sample = pd.read_csv('/share/home/320346/test_df_sample.csv', encoding="utf_8_sig")

# drop index column, 'true_ind' and 'sample' columns
train_df_sample = train_df_sample.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code'
train_df_sample['soc_code'] = train_df_sample['soc_code'].astype(str)
train_df_sample['soc_code1'] = train_df_sample['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
test_df_sample = test_df_sample.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the test set
test_df_sample['soc_code'] = test_df_sample['soc_code'].astype(str)
test_df_sample['soc_code1'] = test_df_sample['soc_code'].map(soc_code_dict)

# drop index column, 'true_ind' and 'sample' columns
valid_df_sample = valid_df_sample.drop(['Unnamed: 0'], axis = 1)
# Generate a new column 'soc_code1' with the mapped values from 'soc_code' for the validation set
valid_df_sample['soc_code'] = valid_df_sample['soc_code'].astype(str)
valid_df_sample['soc_code1'] = valid_df_sample['soc_code'].map(soc_code_dict)




# Tokenize the text and convert it into input features
train_titles = train_df_sample['工作名称'].astype(str).tolist()
train_texts = train_df_sample['工作描述'].astype(str).tolist()
train_labels = train_df_sample['soc_code1'].tolist()


test_titles = test_df_sample['工作名称'].astype(str).tolist()
test_texts = test_df_sample['工作描述'].astype(str).tolist()
test_labels = test_df_sample['soc_code1'].tolist()


valid_titles = valid_df_sample['工作名称'].astype(str).tolist()
valid_texts = valid_df_sample['工作描述'].astype(str).tolist()
valid_labels = valid_df_sample['soc_code1'].tolist()










def assign_weight(true_ind):
    if true_ind:
        return 1.0
    else:
        return 0.5
    
# Assuming you have a column called 'true_ind' in your dataset with boolean values
# True for credible labels and False for less credible labels
train_df_sample['weight'] = train_df_sample['true_ind'].apply(assign_weight)
test_df_sample['weight'] = test_df_sample['true_ind'].apply(assign_weight)
valid_df_sample['weight'] = valid_df_sample['true_ind'].apply(assign_weight)

# Extract the weights
train_weights = train_df_sample['weight'].tolist()
valid_weights = valid_df_sample['weight'].tolist()
test_weights = test_df_sample['weight'].tolist()








# Create the JobPostingDataset class
class JobPostingDataset(Dataset):
    def __init__(self, titles, descriptions, labels, weights, tokenizer, max_length):
        self.titles = titles
        self.descriptions = descriptions
        self.labels = labels
        self.weights = weights
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title = self.titles[idx]
        description = self.descriptions[idx]
        label = self.labels[idx]
        weight = self.weights[idx]

        # Concatenate title and description, repeat the title to give it more importance
        repeat_title = 2  # Adjust this value to control the importance of the title
        text = (title + " ") * repeat_title + description

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Return a tuple of the input tensors, label, and weight
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(label, dtype=torch.long),
            torch.tensor(weight, dtype=torch.float),
        )





# Create the datasets
max_length = 512
train_dataset = JobPostingDataset(train_titles, train_texts, train_labels, train_weights, tokenizer, max_length)
valid_dataset = JobPostingDataset(valid_titles, valid_texts, valid_labels, valid_weights, tokenizer, max_length)
test_dataset = JobPostingDataset(test_titles, test_texts, test_labels, test_weights, tokenizer, max_length)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=20, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)






### Step 2: Model Definition and Steps for Grid Search

# Assuming that train_labels is a list or numpy array of your training labels
# Calculate class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)


# Assuming train_loader, valid_loader, and test_loader are already defined
epochs = 10
best_valid_loss = float('inf')
early_stopping_patience = 3
early_stopping_counter = 0

# LSTM Model Definition
class TextLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, num_classes, bidirectional=True, dropout=0.2):
        super(TextLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, bidirectional=bidirectional, dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # Multiply by 2 for bidirectional
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Hyperparameters (Assuming word_index and other data related variables are already defined)
vocab_size = len(tokenizer)  # Replace with your actual vocab size
embed_dim = 100
num_layers = 2
num_classes = 406  # Replace with your actual number of classes
bidirectional = True

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = class_weights.to(device)

# Define hyperparameter space
hidden_dims = [25, 50, 100]
dropouts = [0.1, 0.2, 0.5]

# Grid search
best_accuracy = 0
best_hyperparams = None
best_model_state = None

for hidden_dim, dropout in itertools.product(hidden_dims, dropouts):
    # Initialize the model
    model = TextLSTM(vocab_size, embed_dim, hidden_dim, num_layers, num_classes, bidirectional, dropout)

    # Check if multiple GPUs are available and wrap the model using nn.DataParallel
    if torch.cuda.device_count() > 1:
        print("Using", torch.cuda.device_count(), "GPUs!")
        model = nn.DataParallel(model)

    # Move the model to the primary device
    model.to(device)
    
    # Define the optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    for epoch in range(epochs):
        train_loss = 0.0
        model.train()
        for batch in train_loader:
            inputs, _, labels, sample_weights = batch
            inputs, labels = inputs.to(device), labels.to(device)
            sample_weights = sample_weights.to(device)  # Move sample weights to the same device as the model

            optimizer.zero_grad()
            
            outputs = model(inputs)  # Forward pass
            loss = criterion(outputs, labels)  # Calculate loss per sample
            weighted_loss = loss * sample_weights  # Apply weights
            final_loss = weighted_loss.mean()  # Take mean of weighted loss
            final_loss.backward()  # Backpropagation
            optimizer.step()
            train_loss += final_loss.item()

        # Validation phase
        valid_loss = 0.0
        correct = 0
        total = 0
        model.eval()
        with torch.no_grad():
            for batch in valid_loader:
                inputs, _, labels, _ = batch  # Unpack and ignore attention mask and weights
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                valid_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        validation_accuracy = correct / total

    # Store the best hyperparameters and model state
    if validation_accuracy > best_accuracy:
        best_accuracy = validation_accuracy
        best_hyperparams = (hidden_dim, dropout)
        best_model_state = model.state_dict()

# Save the best model
print(f"Best Hidden Dim: {best_hyperparams[0]}, Best Dropout: {best_hyperparams[1]}, Best Accuracy: {best_accuracy}")
model_save_path = '/share/home/320346/lstm_model_state_dict.pth'
torch.save({
    'state_dict': best_model_state,
    'hyperparams': best_hyperparams
}, model_save_path)


### Load the trained model (from `HPC`) to perform evaluation
# Assuming you have the model class defined (e.g., TextLSTM)

# Load the trained model and hyperparameters
checkpoint = torch.load('/share/home/320346/lstm_model_state_dict.pth', map_location=torch.device('cpu'))
hidden_dim, dropout = checkpoint['hyperparams']
state_dict = checkpoint['state_dict']

model = TextLSTM(vocab_size, embed_dim, hidden_dim, num_layers, num_classes, bidirectional, dropout)

# Adjust for the keys if it was saved with nn.DataParallel
if list(state_dict.keys())[0].startswith('module.'):
    # Remove 'module.' prefix
    state_dict = {key[len("module."):]: value for key, value in state_dict.items()}

# Load the adjusted state dict
model.load_state_dict(state_dict)




# Create a reverse mapping dictionary to convert the sequential labels back to SOC labels
reverse_soc_code_dict = {v: k for k, v in soc_code_dict.items()}

# Evaluate the model and store predictions
model = model.to(device)
model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for inputs, _, labels, _ in test_loader:  # Ignore attention masks
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        labels = labels.cpu().numpy()

        predictions.extend(preds)
        true_labels.extend(labels)

# Convert predictions and true_labels into NumPy arrays
predictions = np.array(predictions)
true_labels = np.array(true_labels)

# Assuming 'reverse_soc_code_dict' is a dictionary that maps the numeric labels back to the original SOC codes
soc_predictions = [reverse_soc_code_dict[p] for p in predictions]
soc_true_labels = [reverse_soc_code_dict[l] for l in true_labels]

# Compute classification report
unique_labels = sorted(list(set(soc_true_labels).union(set(soc_predictions))))
report = classification_report(soc_true_labels, soc_predictions, labels=unique_labels, output_dict=True)

# Convert report to a pandas DataFrame
report_df = pd.DataFrame(report).transpose()

# Print the report
print(report_df)

# Save the report to a CSV file
report_df.to_csv('/share/home/320346/lstm_report_df.csv', index=True)


#### `Cross model comparison`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Re-importing the dataframes due to a reset in the code execution environment
file_paths = [
    'G:/Data/job_posting/processed/estimation/report_df.csv',
    'G:/Data/job_posting/processed/estimation/bertpass2stage_report_df.csv',
    'G:/Data/job_posting/processed/estimation/lstm_report_df.csv',
    'G:/Data/job_posting/processed/estimation/cnn_report_df.csv',
    'G:/Data/job_posting/processed/estimation/rf_report_df.csv'
]

# Reading the dataframes
dataframes = {path.split('/')[-1]: pd.read_csv(path) for path in file_paths}

# Renaming columns for consistency and adding a 'Model' column
columns_rename = {
    'Unnamed: 0': 'SOC Code', 
    'precision': 'Precision', 
    'recall': 'Recall', 
    'f1-score': 'F1 Score', 
    'support': 'Support'
}

dataframes['report_df.csv'].rename(columns=columns_rename, inplace=True)
dataframes['report_df.csv']['Model'] = 'BERT-WWM Benchmark'

dataframes['bertpass2stage_report_df.csv'].rename(columns=columns_rename, inplace=True)
dataframes['bertpass2stage_report_df.csv']['Model'] = 'BERT-WWM Two-Stage'

dataframes['lstm_report_df.csv'].rename(columns=columns_rename, inplace=True)
dataframes['lstm_report_df.csv']['Model'] = 'LSTM'

dataframes['cnn_report_df.csv'].rename(columns=columns_rename, inplace=True)
dataframes['cnn_report_df.csv']['Model'] = 'CNN'

dataframes['rf_report_df.csv'].rename(columns=columns_rename, inplace=True)
dataframes['rf_report_df.csv']['Model'] = 'RF'

# Combining all dataframes
combined_df = pd.concat(dataframes.values())

# Filtering out the rows that are not occupations (i.e., excluding summary rows)
combined_df = combined_df[combined_df['SOC Code'].apply(lambda x: x.isnumeric())]
overall_avg_df = combined_df.groupby('Model').mean().reset_index()






# Truncating SOC codes to their first 2 digits and calculating averages for each model

# Truncating SOC codes to their first 2 digits and recalculating averages
combined_df['SOC Code 2-Digit'] = combined_df['SOC Code'].str[:2]
# Grouping by the 2-digit SOC code and the model, then calculating the mean
grouped_df_2_digit = combined_df.groupby(['SOC Code 2-Digit', 'Model']).mean().reset_index()


# Calculating the overall average for each metric for each model
overall_avg_df = combined_df.groupby('Model').mean().reset_index()
# Adding a marker for overall averages in the 'SOC Code 4-Digit' column
overall_avg_df['SOC Code 2-Digit'] = 'Average'

# Adding the overall average data to the 2-digit grouped data
final_df_2_digit = pd.concat([overall_avg_df, grouped_df_2_digit], ignore_index=True)
# Reordering the dataframe to have the overall average as the first bar
final_df_2_digit = final_df_2_digit.sort_values(by=['SOC Code 2-Digit', 'Model'])

# Define a set of patterns to use for distinguishing different models
patterns = ['/', '\\', '|', '-', '+']

# Preparing the data for each model separately to apply patterns
models = final_df_2_digit['Model'].unique()
dfs = {model: final_df_2_digit[final_df_2_digit['Model'] == model] for model in models}

# Correcting the issue with bar plotting and reducing the gap between the legend and the figure

# Extracting SOC code labels
soc_code_labels = final_df_2_digit['SOC Code 2-Digit'].unique()

# Setting larger font sizes
plt.rcParams.update({'font.size': 15})  # Adjust this value as needed

# Creating new plots with corrected bar patterns and layout adjustments
plt.figure(figsize=(20, 15))

# Function to plot bars with optimized x-axis range
def plot_with_optimized_x_axis(ax, metric, x_labels):
    # Number of unique SOC codes
    n_soc_codes = len(x_labels)
    # Total number of bars per group
    n_bars = len(models)
    # Calculating the width for each bar
    bar_width = 0.15

    # Positions for each model
    positions = range(n_soc_codes)

    for i, (model, pattern) in enumerate(zip(models, patterns)):
        model_df = final_df_2_digit[final_df_2_digit['Model'] == model]
        # Calculating the x position for each bar
        bar_positions = [x + bar_width * i for x in positions]
        ax.bar(bar_positions, model_df[metric], width=bar_width, label=model, hatch=pattern, edgecolor='black')

    # Adjusting the x-axis range
    ax.set_xlim([-0.5, n_soc_codes - 0])
    mid_positions = [x + bar_width * (len(models) - 1) / 2 for x in positions]
    ax.set_xticks(mid_positions)
    ax.set_xticklabels(x_labels, rotation=0)

# Plotting Precision
ax1 = plt.subplot(3, 1, 1)
plot_with_optimized_x_axis(ax1, 'Precision', soc_code_labels)
ax1.set_title('Panel A: Comparison for Precision (Aggregated at 2-digit SOC)')
ax1.set_ylabel('Precision')
ax1.set_xlabel('SOC Code')

# Plotting Recall
ax2 = plt.subplot(3, 1, 2)
plot_with_optimized_x_axis(ax2, 'Recall', soc_code_labels)
ax2.set_title('Panel B: Comparison for Recall (Aggregated at 2-digit SOC)')
ax2.set_ylabel('Recall')
ax2.set_xlabel('SOC Code')

# Plotting F1 Score
ax3 = plt.subplot(3, 1, 3)
plot_with_optimized_x_axis(ax3, 'F1 Score', soc_code_labels)
ax3.set_title('Panel C: Comparison for F1-score (Aggregated at 2-digit SOC)')
ax3.set_ylabel('F1-score')
ax3.set_xlabel('SOC Code')

# Adjusting the legend
handles, labels = ax3.get_legend_handles_labels()
plt.legend(handles=handles, labels=labels, loc='lower center', bbox_to_anchor=(0.5, -0.35), ncol=len(labels))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()



### Determine the keywords that related to data processing technologies: `keywords_preparing`
- `posting_dataprogfull` is generated: contains firm and posting ID and dummies to identify whether technologies are mentioned in the postings

In [ ]:
# Standard library imports
import json
import os
import re
import urllib3

# Third-party library imports
import cupy as cp
import jieba
import numpy as np
import openai
import pandas as pd
import requests
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor


### Load and merge datasets
# load first data: the UNSPSC classification of software's type
df = pd.read_excel('D:/Dropbox/Dropbox/Paper with Yao-yu/Spatial Data Programming/Data/technology_skill_onet/UNSPSC_reference.xlsx')
# only keep software skills
df = df[df['Family Title'] == 'Software']
df = df.drop(['Commodity Title'], axis = 1)

# load the second data: the software used by each occupation
df1 = pd.read_excel('D:/Dropbox/Dropbox/Paper with Yao-yu/Spatial Data Programming/Data/technology_skill_onet/technology_skills_occupation.xlsx')
# combine the first two datasets, so that we have a occupation title-software-software classification
df3 = pd.merge(df, df1, on = "Commodity Code")
df3 = df3[['O*NET-SOC Code',  'Title', 'Example', 'Hot Technology', 'In Demand', 'Commodity Code', 'Commodity Title', 
           'Class Code', 'Class Title', 'Family Code', 'Family Title', 'Segment Code', 'Segment Title']]
df3 = df3.sort_values(['O*NET-SOC Code', 'Example'], ascending=True)






### Select data-management technology keywords:
# They should be 'Hot-technology'.
# `Class` should be data processing related: 'Class Title' is 'Finance accounting and enterprise resource planning ERP software', or 'Data management and query software', or 'Development software'
# They are in the skill set of data related jobs.
# The main list is based on Burning-glass data, we compensate with Chinese softwares that are obtained from ChatGPT and reviewed by human to check the relevancy.
# Keep only unique values in Name column, and based on the software titles we expand the pool to fit the case in China using ChatGPT

unique_tech = df3.drop_duplicates(subset=['Class Code', 'Class Title', 'Commodity Title', 'Example', 'O*NET-SOC Code', 'Title'])
unique_tech = unique_tech[unique_tech['Title'].str.contains('data|Data')]
# keep if 'Class Title' is 'Finance accounting and enterprise resource planning ERP software', or 'Data management and query software', or 'Development software', or 'Information exchange software'  
unique_tech = unique_tech[unique_tech['Class Title'].isin(['Finance accounting and enterprise resource planning ERP software', 'Data management and query software', 'Development software', 'Industry specific software'])] 
# keep if 'Hot Technology' == 'Y'
unique_tech = unique_tech[unique_tech['Hot Technology'] == 'Y']
unique_tech = unique_tech.loc[:, ['Commodity Title', 'Example', 'Class Code', 'Class Title']]
# keep unique values in Commodity Title
unique_tech = unique_tech.drop_duplicates(subset=['Example'])
# print the unique values in 'class title'
unique_tech['Class Title'].unique()
unique_tech
# keep unique combination of 'Commodity Title' and 'Class Title' 
unique_tech2 = unique_tech.drop_duplicates(subset=['Commodity Title', 'Class Title'])
# keep columns 'Commodity Title' and 'Class Title'
unique_tech2 = unique_tech2.loc[:, ['Commodity Title', 'Class Title']]
# show the entire content in each column 
pd.set_option('display.max_colwidth', None)






#### Manually shorten the title of the software and determine whether it is open source or developed within China (based on `unique_tech`)
# Column `Safe` is determined by human review with dummy 1 if it is open source or developed within China
# read a csv file (includes both Chinese and English characteristics) from 'E:/Data/job_posting/processed/title_short.csv' 
key_df = pd.read_excel('G:/Data/job_posting/processed/title_short.xlsx')
# Generating a new column to represent the combination of 'Commodity Title' and 'Safe'
key_df['Commodity_Safe'] = key_df['Commodity Title'] +  '_' + 'Safe'+  '_' + key_df['Safe'].astype(str)
# Generating a new column to represent the combination of 'Commodity Title' and 'China'
key_df['Commodity_China'] = key_df['Commodity Title'] +  '_' + 'China' + '_' + key_df['China'].astype(str)
key_df
def classify_software(Example, api_key):
    url = 'https://api.openai.com/v1/chat/completions'
    headers = {'Content-Type': 'application/json',
               'Authorization': f'Bearer {api_key}'}
    data = {'model': 'gpt-3.5-turbo-0301',
            'messages':[
                {
                'role': 'user', 
                'content': f'Is this software open source or a software developed in China "{Example}". Please answer with either YES or NO.'
                }
                       ]
            }
    try:
        response = requests.post(url, headers=headers, json=data, verify=False)
        response_data = json.loads(response.text)
        if response.status_code != 200:
            raise Exception(response_data['error']['message'])
        return response_data['choices'][0]['message']['content']
    except Exception as e:
        print(f'Error occurred: {e}')
        return 'N/A'

# Parallelize the job title classification using Python's ThreadPoolExecutor
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

os.environ["http_proxy"] = "http://127.0.0.1:10809"
os.environ["https_proxy"] = "http://127.0.0.1:10809"
openai.organization = "org-VerSGqrvW53HAZizo7yd3d6Z"
api_key = 'sk-**********************************'

# create a thread pool with 30 worker threads
with ThreadPoolExecutor(max_workers=30) as executor:
    # extract the job titles and submit them to the thread pool
    Examples = key_df['Example'].tolist()
    futures = [executor.submit(classify_software, Example, api_key) for Example in Examples]

    # wait for all threads to complete and get the results
    # create an empty list to store soc codes
    Example_short = []
    for future in futures:
        exp_code = future.result()
        Example_short.append(exp_code)

    # append the soc codes to the dataframe
    key_df['Example_short'] = Example_short
key_df.to_csv('E:/Data/job_posting/processed/title_opensource.csv', index = False) 







### Function to check if any word from `工作描述` is present in the `software` column
# This code is run in HPC with 25 CPU cores and 5 GPUs. It took about 4 hours to finish.
def process_file(filename, software_list):
    f = os.path.join(directory, filename)
    df_chunks = pd.read_csv(f, encoding="utf_8_sig", on_bad_lines='skip', usecols=['招聘主键ID', '公司ID', '工作描述'], chunksize=10000)

    output_filename = f.replace('description', 'desp_techfull')
    output_filename = os.path.splitext(output_filename)[0] + "_processed.csv"

    write_header = not os.path.exists(output_filename)
    
    jieba.enable_parallel(4)

    for chunk in df_chunks:
        new_data = cp.zeros((len(chunk), len(software_list)), dtype=cp.int32)
        
        for i, desc in enumerate(chunk['工作描述'].fillna('')):
            words = set(word.lower() for word in jieba.cut(desc, cut_all=False))
            new_data[i, :] = cp.asarray([int(software in words) for software in software_list])
        
        new_df = pd.DataFrame(cp.asnumpy(new_data), columns=software_list)
        
        chunk = pd.concat([chunk.drop(['工作描述'], axis=1).reset_index(drop=True), new_df.reset_index(drop=True)], axis=1)
        chunk.to_csv(output_filename, mode='a', index=False, encoding="utf_8_sig", header=write_header)
        
        write_header = False

    print(f"Processed {filename}")

if __name__ == "__main__":
    directory = '/share/home/320346/description/'
    filenames = [filename for filename in os.listdir(directory) if filename.startswith("job_res_")]

    key_df = pd.read_excel('/share/home/320346/title_short.xlsx')  # Change to read Excel
    key_df = key_df.drop(['Example'], axis=1).rename(columns={'Example_short': 'software'})

    software_list = key_df['software'].str.lower().unique()

    with ProcessPoolExecutor(max_workers=25) as executor:
        executor.map(process_file, filenames, [software_list] * len(filenames))






### Append the data after determining whether it is 'data processing' related, get ready for merging with firm registeration data
# This is done in the HPC with 20 CPU cores and it took about 40 mins to finish.
path = '/share/home/320346/desp_techfull'
files = [f for f in os.listdir(path) if f.endswith('.csv')]

# Initialize an empty DataFrame to hold the final data
dfProgram = pd.DataFrame()

def add_aggregated_columns(df, key_df, classification_column):
    mapping_dict = key_df.set_index('Example_short')[classification_column].to_dict()
    mapping_dict = {k.lower(): v for k, v in mapping_dict.items()}

    # Initialize new columns with zeros
    for unique_value in key_df[classification_column].unique():
        df[unique_value] = 0

    # Summing mentions for each software and filling in the new columns
    for software_col in df.columns.difference(['招聘主键ID', '公司ID']):  # Excluding '招聘主键ID' and '公司ID'
        group_name = mapping_dict.get(software_col, None)
        if group_name:
            df[group_name] += df[software_col]

for file in files:
    file_path = os.path.join(path, file)
    
    # Error handling for file reading
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    df = df[df['招聘主键ID'] != '招聘ID']
    df = df.dropna(subset=['招聘主键ID'])

    df['data_software'] = (df.drop(['招聘主键ID', '公司ID'], axis=1).any(axis=1)).astype(int)

    # Add aggregated columns
    add_aggregated_columns(df, key_df, 'Commodity_Safe')
    add_aggregated_columns(df, key_df, 'Commodity_China')

    # Remove software title columns
    columns_to_remove = df.columns[df.columns.get_loc('quickbooks'):df.columns.get_loc('ajax') + 1]
    df = df.drop(columns=columns_to_remove)

    # Append to the main DataFrame
    # Consider writing to a CSV or using Dask for large datasets
    dfProgram = pd.concat([dfProgram, df], ignore_index=True)

# Summary of the resulting DataFrame
print(dfProgram.info())
dfProgram.to_csv('/share/home/320346/posting_dataprogfull.csv', index=False, encoding='utf-8')

### Get the SOC coded title and extracted data processing softwares in each postings: `firm_data_programming`
- This code is run on HPC. The output is the `spatial_tech_diffusion5`.

In [ ]:
### The classification process adopts mulitple GPUs. The actual process was run on the HPC.
# The path should be adjusted when running the code in HPC.

import os
import pandas as pd
import gc

# Load the fine-tuned model and tokenizer
tokenizer = BertTokenizer.from_pretrained('D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/bert-base-chinese/')
model = BertForSequenceClassification.from_pretrained('F:/Data/job_posting/processed/model/')




# Set the device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
# Check if there are multiple GPUs available
if torch.cuda.device_count() > 1:
    print(f"Let's use {torch.cuda.device_count()} GPUs!")
    # If so, wrap the current model in nn.DataParallel to use multiple GPUs.
    model = torch.nn.DataParallel(model)
# Load the mappings
df = pd.read_csv("F:/Data/job_posting/processed/finetune/est_sample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
df['soc_code'] = df['soc_code'].str.replace('-', '')
unique_soc_codes = sorted(df['soc_code'].unique())
soc_code_dict  = {soc_code: i for i, soc_code in enumerate(unique_soc_codes)}
inverse_soc_code_dict = {v: k for k, v in soc_code_dict.items()}
del df
gc.collect()






### Process job posting data and predict Standard Occupational Classification (SOC) codes for each job posting
# Set the directory where job posting files are located
directory = 'F:/Data/job_posting/mapped_job_posting/Update file/'
# Initialize an empty DataFrame to store processed data
dfPosting = pd.DataFrame()
# Define the batch size for processing data
batch_size = 300  # Adjust as necessary
# Loop through each file in the specified directory
for filename in os.listdir(directory):
    # Process only files that start with "job_res_"
    if filename.startswith("job_res_"):
        # Join the directory path and filename
        f = os.path.join(directory, filename)
        # Read the CSV file, handling encoding and errors
        df = pd.read_csv(f, encoding="utf_8_sig", on_bad_lines='skip', delimiter="?", encoding_errors='ignore')
        # Rename a specific column for consistency
        df.rename(columns={'招聘ID': '招聘主键ID'}, inplace=True)
        # Select only relevant columns from the DataFrame
        df = df[['招聘主键ID', '公司ID', '发布日期', '工作描述', '工作名称', '行业名称']]
        # Set the repetition factor for job titles to emphasize their importance
        repeat_title = 2  # Adjust this value as necessary
        # Create a new column 'input_text' by repeating and concatenating job titles with their descriptions
        df['input_text'] = (df['工作名称'] + ' ') * repeat_title + df['工作描述']
        # Determine the number of examples in the DataFrame
        num_examples = len(df['input_text'])
        # Process the data in batches
        for i in range(0, num_examples, batch_size):
            # Extract a batch of text
            batch_text = df['input_text'][i:i+batch_size].astype(str).tolist()
            # Tokenize the text batch for model input
            inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=512, return_tensors='pt')
            # Move inputs to the specified device (e.g., GPU)
            inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            # Perform model inference without calculating gradients
            with torch.no_grad():
                outputs = model(**inputs)
            # Get predictions from the model outputs
            predictions = torch.argmax(outputs.logits, dim=-1)
            predictions_list = predictions.cpu().numpy().tolist()
            # Map predictions to original SOC codes
            original_soc_codes = [inverse_soc_code_dict[pred] for pred in predictions_list]
            # Store the predicted SOC codes in the DataFrame
            df.loc[i:i+batch_size-1, 'predicted_soc_code'] = original_soc_codes
            # Clear GPU memory cache to optimize performance
            torch.cuda.empty_cache()
        # Drop unneeded columns from the DataFrame
        df.drop(['行业名称', '工作描述', 'input_text'], axis=1, inplace=True)
        # Save the processed DataFrame to a new CSV file
        df.to_csv('F:/Data/job_posting/processed/estimation/{}'.format(filename), index=False, encoding='utf_8_sig')








### Append all the files with the classified occupation titles.
# define the directory
directory = 'G:/Data/job_posting_mapping2/'
# define a list to store DataFrames
dfs = []
# iterate over all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('.csv'):  # check if the file is a csv
        # construct full file path
        file_path = os.path.join(directory, filename)
        # read the csv file and append it to the dfs list
        dfs.append(pd.read_csv(file_path))

# concatenate all DataFrames in the list
dfPosting = pd.concat(dfs, ignore_index=True)
# keep year-month only in the column '发布日期'
dfPosting['发布日期'] = dfPosting['发布日期'].str[:7]

# Number of unique values in the column '发布日期' within each year
# Create a new column that contains the year of each release date
dfPosting['year'] = dfPosting['发布日期'].str[:4]
# Group the DataFrame by year and count the number of unique release dates in each group
result = dfPosting.groupby('year')['发布日期'].nunique()
# Print the result
print(result)
# drop if 'year' is 2020 or 2021
dfPosting = dfPosting[dfPosting['year'] != '2016']
dfPosting = dfPosting[dfPosting['year'] != '2020']
dfPosting = dfPosting[dfPosting['year'] != '2021']
# convert the column '发布日期' to quarterly
dfPosting['发布日期'] = pd.to_datetime(dfPosting['发布日期'])
dfPosting['发布日期'] = dfPosting['发布日期'].dt.to_period('Q')
# drop column '工作名称', '公司ID' 
dfPosting.drop(['工作名称', '公司ID'], axis=1, inplace=True)
# rename 'predicted_soc_code' as 'soc_code'
dfPosting.rename(columns={'predicted_soc_code': 'soc_code'}, inplace=True)









### Load the data with `data programming tech` indicator, and merge with the job posting title data.
# The `data programming tech` indicator is create from keywords_preparing.ipynb.
# `Firm registration data:`
# 'firm_regis_spatial3' largely follows the data cleaning process by 'firm_regis_spatial2'. However, we use the new dataset from RESSET: a more complete firm list (new firm ID indicator to represent same firm changes names), and we have a better way to define branches.
# 'firm_regis_spatial2' is the complete one (seems to be the correct one). 

# The `data programming tech` indicator is create from keywords_preparing.ipynb.
dfProgram = pd.read_csv('G:/Data/job_posting/processed/estimation/posting_dataprogfull.csv', encoding='utf-8')
# merge 'dfProgram' with 'dfPosting' on '招聘主键ID', keep matched sample only
dfPosting = pd.merge(dfPosting, dfProgram, on='招聘主键ID', how='inner')
# generate column 'posting_count` with all values equal to 1
dfPosting['posting_count'] = 1
# Given mapping of commodity names to class titles
commodity_to_class = {
    'Web platform development software': 'Development software',
    'Data base management system software': 'Data management and query software',
    'Data base user interface and query software': 'Data management and query software',
    'Development environment software': 'Development software',
    'Business intelligence and data analysis software': 'Data management and query software',
    'Enterprise application integration software': 'Development software',
    'Computer aided design CAD software': 'Industry specific software',
    'Object or component oriented development software': 'Development software',
    'Configuration management software': 'Development software',
    'Medical software': 'Industry specific software',
    'Object oriented data base management software': 'Data management and query software',
    'Analytical or scientific software': 'Industry specific software',
    'Metadata management software': 'Data management and query software',
    'Program testing software': 'Development software',
    'Enterprise resource planning ERP software': 'Finance accounting and enterprise resource planning ERP software',
    'Data base reporting software': 'Data management and query software',
    'Accounting software': 'Finance accounting and enterprise resource planning ERP software'
}
# Example list of class categories
class_categories = [
    'Finance accounting and enterprise resource planning ERP software',
    'Data management and query software',
    'Development software',
    'Industry specific software'
]

# Create new columns in dfPosting for each class category, initialized to 0
for category in class_categories:
    dfPosting[category] = 0
# Iterate over the commodity_to_class mapping
for commodity, class_category in commodity_to_class.items():
    # Find columns that contain the commodity name
    commodity_columns = [col for col in dfPosting.columns if commodity in col]
    # Create a mask where any of the commodity columns is 1 or more
    mask = dfPosting[commodity_columns].max(axis=1) >= 1
    # Update the corresponding class category column
    dfPosting.loc[mask, class_category] = 1
# Filtering columns based on the specified suffixes ('_Safe_0', '_Safe_1', 'China_0', 'China_1')
suffixes = ['_Safe_0', '_Safe_1', '_China_0', '_China_1']
groups = {suffix: [col for col in dfPosting.columns if col.endswith(suffix)] for suffix in suffixes}

# Creating new columns (ds_safe0, ds_safe1, ds_cn0, ds_cn1) to represent if any column within each group has a value >= 1
# Filling the new columns with dummy variables (0 or 1)
new_columns = {'ds_safe0': '_Safe_0', 'ds_safe1': '_Safe_1', 'ds_cn0': '_China_0', 'ds_cn1': '_China_1'}

# We use the .any() function to quickly check if any value in a row along specified columns is >= 1
# This is more efficient than using .sum() on large datasets
for new_col, suffix in new_columns.items():
    dfPosting[new_col] = dfPosting[groups[suffix]].any(axis=1).astype(int)

# Create new columns for storing the sum of selected columns
new_columns_sum = {'ds_safe0_sum': '_Safe_0', 'ds_safe1_sum': '_Safe_1', 'ds_cn0_sum': '_China_0', 'ds_cn1_sum': '_China_1'}
for new_col, suffix in new_columns_sum.items():
    dfPosting[new_col] = dfPosting[groups[suffix]].sum(axis=1)
# Create a new column 'data_software_sum' to record the sum of the newly generated columns
sum_columns = list(new_columns_sum.keys())
dfPosting['data_software_sum'] = dfPosting[sum_columns].sum(axis=1)
# Removing all columns related to software skills that end with '_Safe_0', '_Safe_1', 'China_0', or 'China_1'
columns_to_remove = [col for suffix in suffixes for col in groups[suffix]]
dfPosting.drop(columns=columns_to_remove, inplace=True)
# export to ''G:/Data/job_posting/processed/estimation/posting_program.csv'' 
dfPosting.to_csv('G:/Data/job_posting/processed/estimation/posting_program.csv', index=False, encoding='utf-8')









### Load the `firm registration` data, and merge with the `dfPosting`, and export the final data for future usage.
# `firm registration` data is generated from `firm_reg_v2.ipynb`
filtered_dataframe2 = pd.read_csv('G:/Data/job_posting/processed/estimation/firm_regis_spatial3.csv', encoding='utf-8')
# drop '经营范围_businessScope', '企业名称_companyName' to save memory
filtered_dataframe2.drop(['企业名称_companyName', '企业类型'], axis=1, inplace=True)
# rename '公司主键_companyId' as '公司ID'
filtered_dataframe2.rename(columns={'公司主键_companyId': '公司ID'}, inplace=True)
# merge 'filtered_dataframe2' and 'dfPosting' on '公司ID', keep matched and observations from 'filtered_dataframe2'
filtered_dataframe2 = pd.merge(filtered_dataframe2, dfPosting, on='公司ID', how='left')
# The master dataframe `filtered_dataframe2` includes firm registration, standardized occupation code, and the `data programming tech` indicator.
# simple data cleaning: drop observations with `省份_province` cannot be identified.
filtered_dataframe2 = filtered_dataframe2[filtered_dataframe2['省份_province'].notnull()]








#### `spatial_tech_diffusion5` has been generated:
# `spatial_tech_diffusion5` is the merged 'posting_dataprogfull' and 'firm_regis_spatial3', incorporating the classes of the technologies. 
filtered_dataframe2.to_csv('G:/Data/job_posting/processed/estimation/spatial_tech_diffusion5.csv', index=False, encoding='utf_8_sig')

### Processing the firm registration data: `firm_reg_v2`
- `firm_regis_spatial3` is generated: contains firm ID, industry, ownership and so on

In [ ]:
# We have `new_com_id`: 公司主键companyid 是一个公司的唯一主键，如果公司变更名称，那么数据库中会增加一条新名称的数据记录，生成新的companyid，同时会在原companyid对应的数据记录中的new_com_id字段标识出新生成的companyid
# We have a better way to link the parent firm and branches. The data is from an exteral source purchased from RESSET
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np
from datetime import datetime
import gc, json, csv, re, os, glob

### Load the raw data and process in chunks due to the memory limit.
# filter1: exclude the '核准日期_approvedTime' is later than 2020 (before the COVID), or missing
# filter2: drop firms registered and out of market before 2016
# filter3: drop observations that the industry category is '其他' or missing and cannot be backed out from '经营范围_businessScope'
# filter4: `公司主键_companyId` is missing
# filter5: drop if the firm type is `个体`

def classify_status(status):
    closed_statuses = ['注销', '吊销未注销', '吊销', '吊销并注销', '撤销', '已注销', '注销企业', '注吊销', '吊销,已注销', ' 撤销', '已撤销登记', 
    '注销(简易)', '吊销，未注销', '吊销企业', ' 吊销，未注销', '已吊销', '非正常户', '吊销，已注销', ' 注销',  '清算中',  '停业', '吊销,未注销', '歇业', '已告解散']
    run_statuses = ['正常', '存续', '其他', '存续（在营、开业、在册）', '在营', '开业', '在营（开业）企业', ' 存续（在营、开业、在册）', '在营（开业）', '在业', 
    '登记成立', '存续(在营、开业、在册)',  '已开业', '登记', '开业（存续）', '仍注册']
    
    if str(status).strip() in closed_statuses:
        return 'closed'
    elif str(status).strip() in run_statuses:
        return 'run'
    else:
        return 'unknown'

chunk_size = 10000000  # Adjust based on your system's memory
output_file_path = "G:/Data/firm_regist_2purchase/工商信息.csv"

# Define a reader for reading the file in chunks
reader = pd.read_csv(output_file_path, chunksize=chunk_size, encoding='utf-8', delimiter=",")
# Define the path for the processed file
processed_file_path = 'F:/Data/firm_regis/new_id_firm.csv'
# Iterate through the chunks
for chunk in reader:
    # Keep observations with numerical values in specified columns
    chunk = chunk[pd.to_numeric(chunk['公司主键_companyId'], errors='coerce').notnull()]
    chunk = chunk[pd.to_numeric(chunk['新公司主键New_id'], errors='coerce').notnull()]
    # Drop duplicates by '公司主键_companyId'
    chunk = chunk.drop_duplicates(subset=['公司主键_companyId'], keep='first')
    # Optional: Filter based on '注册时间_estiblishTime' if this column is present in the chunk
    chunk = chunk[chunk['注册时间_estiblishTime'] < '2020-01-01']
    chunk = chunk[(chunk['注册时间_estiblishTime'].notnull()) | (chunk['核准日期_approvedTime'].notnull())]
    chunk = chunk[(chunk['注册时间_estiblishTime'] != '--') | (chunk['核准日期_approvedTime'] != '--')]
    # Create a new variable 'firm_status' representing the firm's status
    chunk['firm_status'] = chunk['经营状态_regStatus'].apply(classify_status)
    # Filter rows where 'firm_status' is not 'closed' or '注册时间_estiblishTime' is 2016 or later
    chunk = chunk.drop(chunk[(chunk['firm_status'] == 'closed') & (chunk['核准日期_approvedTime'] < '2017-01-01')].index)
    # drop rows with '个体' in column '企业类型_companyOrgType'
    chunk['企业类型_companyOrgType'] = chunk['企业类型_companyOrgType'].fillna('')
    chunk = chunk[~chunk['企业类型_companyOrgType'].str.contains('个体')]
    # drop column '注册时间_estiblishTime'
    chunk = chunk.drop(['法定代表人_legalPersonName', '统一社会信用代码_uscCode', '注册资本_regCapital'], axis=1)
    # Append the processed chunk to the new file (use mode='a' to append)
    chunk.to_csv(processed_file_path, mode='a', index=False)











### There are several issues need to be addressed later:
# The '公司主键_companyId' is not a unique identifier for each company. Different values will be assigned if the company changes name or have a new '统一社会信用代码_uscCode'. This is troublesome when we decide whether firm is out of market. We need to acquire the firm registration status data to determine whether the firm is still in market. (`SOLVED`: new data, using `new_com_id`)
# There are a significant amount of firms with missing industrial code. We need to classifiy those firms into '一级行业_categoryStrBig' and '二级行业_categoryStrMiddle' by using the '经营范围_businessScope'. (`SOLVED`: using a BERT model to classify the missing values)
# Our current method to determine the branch and associated parent firm is based on whether the branch's '企业名称_companyName' is contained in parent firm's '企业名称_companyName'. However, this method is not reliable. We need to acquire the 'branch-parent' dataset to determine whether the firm is a branch or a parent company. (`SOLVED`: new data, we get new data from RESSET)

# Read the processed file for further analysis
combined_dataframe = pd.read_csv('F:/Data/firm_regis/new_id_firm.csv', encoding='utf-8', on_bad_lines='skip', encoding_errors='ignore')
# convert '公司主键_companyId' to string
combined_dataframe['公司主键_companyId'] = combined_dataframe['公司主键_companyId'].astype(str)
# drop if '注册时间_estiblishTime' == '--' or '核准日期_approvedTime' == '--'
combined_dataframe = combined_dataframe[combined_dataframe['注册时间_estiblishTime'] != '--']
combined_dataframe = combined_dataframe[combined_dataframe['核准日期_approvedTime'] != '--']

# Filter: drop observations that the industry category is '其他' or missing and cannot be backed out from '经营范围_businessScope': drop if column '一级行业_categoryStrBig' is '--' or '其他' or null, at the same time the column '经营范围_businessScope' is '--' or '其他' or null
combined_dataframe = combined_dataframe.drop(
    combined_dataframe[
        (combined_dataframe['一级行业_categoryStrBig'].isin(['--', '其他']) | combined_dataframe['一级行业_categoryStrBig'].isnull()) &
        (combined_dataframe['经营范围_businessScope'].isin(['--', '其他']) | combined_dataframe['经营范围_businessScope'].isnull())
    ].index
)

# drop duplicates by column '公司主键_companyId', keep the first one
combined_dataframe = combined_dataframe.drop_duplicates(subset=['公司主键_companyId'], keep='first')
# Before continue:
# total number of observations
print(combined_dataframe.shape[0])










### A. Classifiy the firms into `一级行业_categoryStrBig` and `二级行业_categoryStrMiddle` when the industry information is missing
# This code block is used to convert the 2011 version industrial classification to the 2017 version.

# replace values in '二级行业_categoryStrMiddle': '广播、电视、电影和影视录音制作业' -> '广播、电视、电影和录音业', '建筑装饰和其他建筑业' -> '建筑装饰、装修和其他建筑业', 
# '仓储业' -> '装卸搬运和仓储业', '装卸搬运和运输代理业' -> '多式联运和运输代理业', '农、林、牧、渔服务业' -> '农、林、牧、渔专业及辅助性活动', 
# '开采辅助活动' -> '开采专业及辅助性活动', '石油加工、炼焦和核燃料加工业' -> '石油、煤炭及其他燃料加工业'
combined_dataframe['二级行业_categoryStrMiddle'] = combined_dataframe['二级行业_categoryStrMiddle'].replace(['广播、电视、电影和影视录音制作业', '建筑装饰和其他建筑业', '仓储业', '装卸搬运和运输代理业', 
                                                '农、林、牧、渔服务业', '开采辅助活动', '石油加工、炼焦和核燃料加工业'], ['广播、电视、电影和录音业', '建筑装饰、装修和其他建筑业', '装卸搬运和仓储业', 
                                                '多式联运和运输代理业', '农、林、牧、渔专业及辅助性活动', '开采专业及辅助性活动', '石油、煤炭及其他燃料加工业'])

# To get the frequencies of each value of '二级行业_categoryStrMiddle' and sort them in ascending order
freq = combined_dataframe['二级行业_categoryStrMiddle'].value_counts().sort_values()
## Re-load the data back once the calculation is done based on code: `indsector_classification.ipynb`
dfMissing = pd.read_csv('G:/Data/job_posting/processed/estimation/predict_indSector.csv', encoding='utf-8')
# drop columns '一级行业_categoryStrBig', '二级行业_categoryStrMiddle'
dfMissing = dfMissing.drop(['一级行业_categoryStrBig', '二级行业_categoryStrMiddle'], axis=1)
# rename column 'predicted_ind_code' to '二级行业_categoryStrMiddle' 
dfMissing = dfMissing.rename(columns={'predicted_ind_code': '二级行业_categoryStrMiddle'})
# create a unique mapping (dictionary) between '一级行业_categoryStrBig' and '二级行业_categoryStrMiddle' from the 'combined_dataframe'.
mapping_dict = combined_dataframe.set_index('二级行业_categoryStrMiddle')['一级行业_categoryStrBig'].to_dict()
# use this dictionary to map the '二级行业_categoryStrMiddle' in 'dfMissing' to generate the missing '一级行业_categoryStrBig'.
dfMissing['一级行业_categoryStrBig'] = dfMissing['二级行业_categoryStrMiddle'].map(mapping_dict)
# drop if '注册时间_estiblishTime' == '--' or '核准日期_approvedTime' == '--'
dfMissing = dfMissing[dfMissing['注册时间_estiblishTime'] != '--']
dfMissing = dfMissing[dfMissing['核准日期_approvedTime'] != '--']
# convert '公司主键_companyId' to string
dfMissing['公司主键_companyId'] = dfMissing['公司主键_companyId'].astype(int).astype(str)
# drop observations in 'combined_dataframe' if '二级行业_categoryStrMiddle' is '--' or '其他' or null, at the same time the column '经营范围_businessScope' is NOT '--' or '其他' or null
combined_dataframe = combined_dataframe[
    (~combined_dataframe['二级行业_categoryStrMiddle'].isin(['--', '其他']) | combined_dataframe['二级行业_categoryStrMiddle'].isnull()) |
    (combined_dataframe['经营范围_businessScope'].isin(['--', '其他']) | combined_dataframe['经营范围_businessScope'].isnull())
]
# append 'dfMissing' to 'combined_dataframe'
combined_dataframe = pd.concat([combined_dataframe, dfMissing], ignore_index=True)
print(combined_dataframe.shape)







### B. Standarize values in columns '省份_province', '城市_city'
province_mapping = {
    '国家': '国家',
    '北京市': '北京',
    '河南省': '河南',
    '天津市': '天津',
    '河北省': '河北',
    '江苏省': '江苏',
    '黑龙江省': '黑龙江',
    '吉林省': '吉林',
    '广东省': '广东',
    '新疆维吾尔自治区': '新疆',
    '海南省': '海南',
    '辽宁省': '辽宁',
    '湖北省': '湖北',
    '四川省': '四川',
    '浙江省': '浙江',
    '福建省': '福建',
    '重庆市': '重庆',
    '江西省': '江西',
    '湖南省': '湖南',
    '陕西省': '陕西',
    '山东省': '山东',
    '山西省': '山西',
    '上海市': '上海',
    '西藏自治区': '西藏',
    '安徽省': '安徽',
    '广西壮族自治区': '广西',
    '内蒙古自治区': '内蒙古',
    '青海省': '青海',
    '贵州省': '贵州',
    '甘肃省': '甘肃',
    '云南省': '云南',
    '宁夏回族自治区': '宁夏',
    '香港特别行政区': '香港',
    '台湾': '台湾',
    '新疆': '新疆',
    '广西': '广西',
    '内蒙古': '内蒙古',
    '宁夏': '宁夏',
    '新疆省': '新疆',
    '中国贵州省': '贵州',
    '香港': '香港',
    '黑龙省': '黑龙江',
    '黒龙江省': '黑龙江',
    '龙江省': '黑龙江',
    '甘肃 省': '甘肃',
    '广西省': '广西',
    '省': 'unknown',
    '广西自治区': '广西',
    'i浙江省': '浙江',
    '洒北省': 'unknown',
    '何南省': 'unknown',
    '黑龙江佳木斯市省': '黑龙江',
    '江本省': 'unknown'
}

def find_province(province_str):
    for key, value in province_mapping.items():
        if key in str(province_str):
            return value
    return 'unknown'
# Create a new variable 'province' based on the given list of strings
combined_dataframe['省份_province'] = combined_dataframe['省份_province'].apply(find_province)
# unique values in '省份_province' and frequency for a manual check
print(combined_dataframe['省份_province'].value_counts())
# Define a regex pattern for province extraction, including special cases
province_pattern = r'(.*?省|北京市|天津市|上海市|重庆市)'
combined_dataframe['province_extracted'] = combined_dataframe['注册地址_regLocation'].str.extract(province_pattern)
# Define a regex pattern for city extraction, including special rules for specified provinces
city_pattern = r'(?:[省]|北京市|天津市|上海市|重庆市)(.*?[市区])'
combined_dataframe['city_extracted'] = combined_dataframe['注册地址_regLocation'].str.extract(city_pattern)
# Show the results
combined_dataframe[['注册地址_regLocation', 'province_extracted', 'city_extracted']].head()
# Remove '省' from the city column
combined_dataframe['province_extracted'] = combined_dataframe['province_extracted'].str.replace('省', '')
# drop column '注册地址_regLocation' in 'filtered_dataframe1' to save memory
combined_dataframe = combined_dataframe.drop(['注册地址_regLocation'], axis=1)
# Update the values in the '省份_province' and '城市_city' columns if '省份_province' takes the value '国家' or 'unknown', replacing them with the values from the 'province' and 'city' columns respectively.
combined_dataframe.loc[combined_dataframe['省份_province'].isin(['国家', 'unknown']), '城市_city'] = combined_dataframe.loc[combined_dataframe['省份_province'].isin(['国家', 'unknown']), 'city_extracted']
combined_dataframe.loc[combined_dataframe['省份_province'].isin(['国家', 'unknown']), '省份_province'] = combined_dataframe.loc[combined_dataframe['省份_province'].isin(['国家', 'unknown']), 'province_extracted']
# drop column 'province' and 'city'
combined_dataframe.drop(['province_extracted', 'city_extracted'], axis=1, inplace=True)
# List of specified provinces
special_provinces = ['北京', '天津', '上海', '重庆']
# Condition to target rows with specified provinces
condition = combined_dataframe['省份_province'].isin(special_provinces)
# Update the '城市_city' column with values from the '省份_province' column, appending '市' for the specified provinces
combined_dataframe.loc[condition, '城市_city'] = combined_dataframe.loc[condition, '省份_province'] + '市'










### C. Standarize values in column '企业类型_companyOrgType'
# read a xlsx file, this is the firm registeration type dictionary created by the government. We downloaded from 'http://tjj.beijing.gov.cn/zwgkai/tjbz_31390/qttjfl_31393/202002/t20200214_1631949.html'
firm_regtype = pd.read_excel("F:/Data/firm_regis/firm_type.xlsx")
# check unique values in '企业类型' column
firm_regtype['企业类型'].unique()
#### After loading the dataset, we feed the classification into GPT3.5 and ask it to do a preliminary classification
#### After GPT3.5 returns the clasification:
# we clean the column 'mapped_type'
# we manually check and correct for the misclassified observations
# we merge 'filtered_df2' with 'firm_regtype' on '企业类型' column, now most of the observations are matched

# Next, we manually check and correct for the misclassified observations
# we load the data back after the manual check classification 
filtered_df2 = pd.read_excel("F:/Data/firm_regis/firm_type_mapped.xlsx")
# rename '代码' as '企业类型代码_matched' to recode '企业类型_companyOrgType' is exactly matched with governmental defined type
filtered_df2.rename(columns={'代码':'企业类型代码_matched'}, inplace=True)
# replace 'mapped_type' with '企业类型' if '代码' is not null
filtered_df2.loc[filtered_df2['企业类型代码_matched'].notnull(), 'mapped_type'] = filtered_df2['企业类型']
# rename '企业类型' to '企业类型_companyOrgType', and rename 'mapped_type' to '企业类型'
filtered_df2.rename(columns={'企业类型':'企业类型_companyOrgType', 'mapped_type':'企业类型'}, inplace=True)
# merge 'filtered_df2' with 'firm_regtype' on '企业类型' column, now most of the observations are matched
filtered_df3 = pd.merge(filtered_df2, firm_regtype, on='企业类型', how='left')
# merge 'combined_dataframe' with 'filtered_df3' on '企业类型_companyOrgType' column, only keep matched observations
combined_dataframe = pd.merge(combined_dataframe, filtered_df3, on='企业类型_companyOrgType', how='left')
# rename '企业类型' to '企业类型_companyOrgType', and rename 'mapped_type' to '企业类型'
combined_dataframe.rename(columns={'代码':'企业类型代码'}, inplace=True)
# drop column '企业类型_companyOrgType', '经营状态_regStatus' and '法定代表人ID_legalPersonName' 
combined_dataframe.drop(columns=['经营状态_regStatus', '经营范围_businessScope'], inplace=True)












### D. Identify branch, and relate them to their parent company
Dfbranch = pd.read_csv('F:/Data/firm_regis/branch_association.csv', encoding='utf_8')
# convert '公司主键_companyId' to string
Dfbranch['公司主键_companyId'] = Dfbranch['公司主键_companyId'].astype(int).astype(str)
Dfbranch['parent_firm'] = Dfbranch['parent_firm'].astype(int).astype(str)
# clean all the spaces in the string
combined_dataframe['公司主键_companyId'] = combined_dataframe['公司主键_companyId'].str.strip()
Dfbranch['公司主键_companyId'] = Dfbranch['公司主键_companyId'].str.strip()
# merge 'Dfbranch' with 'combined_dataframe' on '公司主键_companyId'
combined_dataframe = pd.merge(combined_dataframe, Dfbranch, how='left', left_on='公司主键_companyId', right_on='公司主键_companyId')
# Create conditions for the 'branch', 'parent', and 'local' columns
condition_branch = (combined_dataframe['parent_firm'].notna()) & (combined_dataframe['公司主键_companyId'] != combined_dataframe['parent_firm'])
condition_parent = (combined_dataframe['parent_firm'].notna()) & (combined_dataframe['公司主键_companyId'] == combined_dataframe['parent_firm'])
condition_local = combined_dataframe['parent_firm'].isna()
# Assign values based on the conditions
combined_dataframe['branch'] = condition_branch.astype(int)
combined_dataframe['parent'] = condition_parent.astype(int)
combined_dataframe['local'] = condition_local.astype(int)
# Rename the column 'parent_firm' to 'parent_firm_id'
combined_dataframe.rename(columns={'parent_firm': 'parent_firm_id'}, inplace=True)
# Replace 'parent_firm_id' with '公司主键_companyId' if 'parent_firm_id' is null
combined_dataframe['parent_firm_id'] = combined_dataframe['parent_firm_id'].where(combined_dataframe['parent_firm_id'].notna(), combined_dataframe['公司主键_companyId'])








### Drop unnecessary columns and save the dataset, get ready to merge with the job posting data
# drop if '公司主键_companyId' is duplicated
combined_dataframe.drop_duplicates(subset=['公司主键_companyId'], inplace=True)
# check whether '公司主键_companyId' is unique
print(combined_dataframe['公司主键_companyId'].is_unique)
# export to 'F:/Data/job_posting/processed/estimation'
combined_dataframe.to_csv('G:/Data/job_posting/processed/estimation/firm_regis_spatial3.csv', index=False, encoding='utf-8')

#### Processing the firm industry classification using the same Chinese BERT-wwm model: `indsector_classification`

In [ ]:
## All the training and calculation is done on HPC, this is the archive of the code
from transformers import BertForSequenceClassification, AdamW, get_scheduler, Trainer, BertTokenizer, get_linear_schedule_with_warmup
import torch
import numpy as np
import pandas as pd
import csv
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score

tokenizer = BertTokenizer.from_pretrained("D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/chinese-bert-wwm/")
df = pd.read_csv('G:/Data/job_posting/processed/estimation/firm_sectorSample.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
batch_size = 128

# Generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order. Create a dictionary to map unique soc_codes to sequential integer labels
unique_ind_codes = sorted(df['二级行业_categoryStrMiddle'].unique())
ind_code_dict  = {二级行业_categoryStrMiddle: i for i, 二级行业_categoryStrMiddle in enumerate(unique_ind_codes)}

# create into train, validation and test set
train_df_sample, temp_df_sample = train_test_split(df, test_size=0.4, random_state=42)
valid_df_sample, test_df_sample = train_test_split(temp_df_sample, test_size=0.5, random_state=62)

# export the train, validation and test set to csv
train_df_sample.to_csv('G:/Data/job_posting/processed/finetune/train_df_Indsample.csv', index=False, encoding = "utf_8_sig", header=True)
test_df_sample.to_csv('G:/Data/job_posting/processed/finetune/test_df_Indsample.csv', index=False, encoding = "utf_8_sig", header=True)
valid_df_sample.to_csv('G:/Data/job_posting/processed/finetune/valid_df_Indsample.csv', index=False, encoding = "utf_8_sig", header=True)


# Generate a new column 'ind_code1' with the mapped values from '二级行业_categoryStrMiddle'
train_df_sample['ind_code1'] = train_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)
# Generate a new column 'ind_code1' with the mapped values from 'soc_code' for the test set
test_df_sample['ind_code1'] = test_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)
# Generate a new column 'ind_code1' with the mapped values from 'soc_code' for the validation set
valid_df_sample['ind_code1'] = valid_df_sample['二级行业_categoryStrMiddle'].map(ind_code_dict)

# Tokenize the text and convert it into input features
train_texts = train_df_sample['经营范围_businessScope'].astype(str).tolist()
train_labels = train_df_sample['ind_code1'].tolist()

test_texts = test_df_sample['经营范围_businessScope'].astype(str).tolist()
test_labels = test_df_sample['ind_code1'].tolist()

valid_texts = valid_df_sample['经营范围_businessScope'].astype(str).tolist()
valid_labels = valid_df_sample['ind_code1'].tolist()

# Create the JobPostingDataset class
class JobPostingDataset(Dataset):
    def __init__(self, descriptions, labels, tokenizer, max_length):
        self.descriptions = descriptions
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        description = self.descriptions[idx]
        label = self.labels[idx]

        # Tokenize the text
        encoding = self.tokenizer.encode_plus(
            description,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Return a tuple of the input tensors, label, and weight
        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(label, dtype=torch.long),
        )

# Create the datasets
max_length = 512
train_dataset = JobPostingDataset(train_texts, train_labels, tokenizer, max_length)
valid_dataset = JobPostingDataset(valid_texts, valid_labels, tokenizer, max_length)
test_dataset = JobPostingDataset(test_texts, test_labels, tokenizer, max_length)

# Create the data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
def evaluate(model, valid_loader, device, loss_fn):
    model.eval()
    total_loss = 0
    num_batches = 0
    with torch.no_grad():
        for batch in valid_loader:
            inputs = batch[0].to(device)
            masks = batch[1].to(device)
            labels = batch[2].to(device)

            logits = model(inputs, attention_mask=masks).logits
            batch_loss = loss_fn(logits, labels)
            loss = torch.mean(batch_loss)

            total_loss += loss.item()
            num_batches += 1
    return total_loss / num_batches







# Define the model, the optimizer, and the learning rate scheduler
num_labels = len(train_df_sample['ind_code1'].unique())
model = BertForSequenceClassification.from_pretrained("D:/Dropbox/Dropbox/vs_cloud/Job_posting_data/chinese-bert-wwm/", num_labels=num_labels)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
num_epochs = 20
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
# Compute class weights using the train_labels_np array
unique_labels = train_df_sample['ind_code1'].unique()
class_weights = compute_class_weight('balanced', classes=unique_labels, y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float) 
# The patience parameter determines how many consecutive epochs the model can go without an improvement in validation loss before stopping the training. 
# In this case, the patience is set to 3, meaning that if the validation loss does not improve for 3 consecutive epochs, the training will be stopped.
early_stopping_patience = 3
# This line initializes a counter variable called num_epochs_without_improvement that keeps track of the number of consecutive epochs without an improvement in validation loss. 
# The counter is set to 0 at the beginning of the training process and is incremented by 1 whenever there is no improvement in the validation loss. If the validation loss improves in a particular epoch, the counter is reset to 0.
num_epochs_without_improvement = 0
# During the training loop, if num_epochs_without_improvement becomes equal to or greater than early_stopping_patience, the training will be stopped. 
# This way, the training process can be terminated early when the model starts overfitting, or when there is no significant improvement in the validation loss.
best_valid_loss = float('inf')

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# Initialize lists to store losses
train_losses = []
# Utilize multiple GPUs with DataParallel
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
# Pass the computed class_weights to the CrossEntropyLoss function:
# Create a loss function that doesn't reduce the losses right away and pass class_weights
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(device), reduction='none')
# By incorporating class weights into the loss function, the model will pay more attention to the minority classes during training. 

model.train()
for epoch in range(num_epochs):
    epoch_train_loss = 0
    num_batches = 0
    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()

        logits = model(inputs, attention_mask=masks).logits

        # Compute the loss for each sample
        batch_loss = loss_fn(logits, labels)

        # Average the weighted losses
        loss = torch.mean(batch_loss)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        # Add the current batch loss to the epoch_train_loss
        epoch_train_loss += loss.item()
        num_batches += 1

    # Calculate average loss for the current epoch and append it to the train_losses list
    epoch_train_loss /= num_batches
    train_losses.append(epoch_train_loss)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {epoch_train_loss:.4f}")

    # Evaluate the model on the validation set
    valid_loss = evaluate(model, valid_loader, device, loss_fn)
    print(f"Validation Loss: {valid_loss:.4f}")

    # Save the best model based on the validation loss
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        if isinstance(model, torch.nn.DataParallel):
            model.module.save_pretrained("G:/Data/job_posting/processed/finetune/best_model_Indsample")
        else:
            model.save_pretrained("G:/Data/job_posting/processed/finetune/best_model_Indsample")
        num_epochs_without_improvement = 0
    else:
        num_epochs_without_improvement += 1

    # Check the stopping condition and break the loop if needed
    if num_epochs_without_improvement >= early_stopping_patience:
        print("Early stopping due to no improvement in validation loss.")
        break
actual_num_epochs = len(train_losses)









# Evaluate the performance of the model on the test set
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)

        # Resize the attention mask tensor to match the size of the inputs
        # masks.resize_(inputs.shape[0], inputs.shape[1])
    
        outputs = model(inputs, attention_mask=masks)
        logits = outputs.logits
        batch_predictions = torch.argmax(logits, axis=1).cpu().numpy()
        predictions.extend(batch_predictions)

accuracy = accuracy_score(test_labels, predictions)
print(f"Accuracy: {accuracy}")

# First, we need to modify the true_labels and predictions by extracting the major SOC groups from the original SOC codes.
# Create a list of SOC codes ordered by their corresponding sequential labels
ordered_ind_codes = [二级行业_categoryStrMiddle for 二级行业_categoryStrMiddle, _ in sorted(ind_code_dict.items(), key=lambda item: item[1])]

# Convert test_labels and predictions back to SOC codes
test_labels_ind = np.array([ordered_ind_codes[label] for label in test_labels])
predictions_ind = np.array([ordered_ind_codes[label] for label in predictions])

# Calculate accuracy for each major SOC group
major_group_accuracies = {}
unique_major_groups = np.unique(test_labels_ind)
for major_group in unique_major_groups:
    major_group_indices = np.where(test_labels_ind == major_group)
    major_group_accuracy = accuracy_score(test_labels_ind[major_group_indices], predictions_ind[major_group_indices])
    major_group_accuracies[major_group] = major_group_accuracy
    print(f"Major Group {major_group}: Accuracy: {major_group_accuracy:.4f}")

major_group_accuracies_df = pd.DataFrame(list(major_group_accuracies.items()), columns=['Major Group', 'Accuracy'])
major_group_accuracies_df.to_csv('/share/home/320346/trained_model/Indmajor_group_accuracies.csv', index=False)
### After the training is done, we call the predict function to predict the class of the data
import os
import pandas as pd
import torch
import gc
from transformers import BertTokenizer, BertForSequenceClassification

# Load the fine-tuned model and tokenizer
tokenizer = BertTokenizer.from_pretrained('/share/home/320346/bert-base-chinese/')
model = BertForSequenceClassification.from_pretrained('/share/home/320346/best_model_Indsample/')
# Set the device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# Check if there are multiple GPUs available
if torch.cuda.device_count() > 1:
    print(f"Let's use {torch.cuda.device_count()} GPUs!")
    # If so, wrap the current model in nn.DataParallel to use multiple GPUs.
    model = torch.nn.DataParallel(model)

# Load the mappings
df = pd.read_csv("/share/home/320346/firm_sectorSample.csv", encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
unique_ind_codes = sorted(df['二级行业_categoryStrMiddle'].unique())
ind_code_dict  = {二级行业_categoryStrMiddle: i for i, 二级行业_categoryStrMiddle in enumerate(unique_ind_codes)}
inverse_ind_code_dict = {v: k for k, v in ind_code_dict.items()}
del df

batch_size = 1200  # Adjust as necessary
df = pd.read_csv('/share/home/320346/mapped_ind_sector/sectorMissing.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
num_examples = len(df['经营范围_businessScope'])

for i in range(0, num_examples, batch_size):
    batch_text = df['经营范围_businessScope'][i:i+batch_size].astype(str).tolist()
    inputs = tokenizer(batch_text, padding=True, truncation=True, max_length=512, return_tensors='pt')

    inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
            
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)
    predictions_list = predictions.cpu().numpy().tolist()
    original_ind_codes = [inverse_ind_code_dict[pred] for pred in predictions_list]
    df.loc[i:i+batch_size-1, 'predicted_ind_code'] = original_ind_codes
    torch.cuda.empty_cache()  
df.to_csv('/share/home/320346/predict_indSector.csv', index=False, encoding='utf_8_sig')

### Replicate the results in the manuscript: `data_spatial_dependency`

##### The following code is needed for running the `data_spatial_dependency`
- `io_table`
- `yearbook_data`
- `China_US_comparison`

In [ ]:
### Load the dataframe `filtered_dataframe2` created from `firm_data_programming.ipynb`
# Important to note that `filtered_dataframe2` is at year-firm-posting level. Thus, the firm ID is not unique.
import os
import pandas as pd
import gc
import numpy as np

filtered_dataframe2 = pd.read_csv('G:/Data/job_posting/processed/estimation/spatial_tech_diffusion5.csv', encoding='utf-8')
filtered_dataframe2.head()
## This step is used for adjusting the sampling bias in the data at `'省份_province, 城市_city, year, 一级行业_categoryStrBig'`
# Because the sampling adjustment is based on the new employment by industrial sectors, we first set the observation with job posting data but missing industrial sector information to be null. 
# This is because we cannot use the job posting data to adjust the sampling bias if we do not know the industrial sector of the job posting.
# replace '其他' in column '一级行业_categoryStrBig' with '\\N'
filtered_dataframe2['一级行业_categoryStrBig'] = filtered_dataframe2['一级行业_categoryStrBig'].replace('其他', '\\N')
# unique values in '一级行业_categoryStrBig' of filtered_dataframe2
filtered_dataframe2['一级行业_categoryStrBig'].unique()
# replace columns 'year', '招聘主键ID', '发布日期', 'soc_code', 'software_data', 'posting_count', '一级行业_categoryStrBig' with null if '一级行业_categoryStrBig' == '\\N'
#filtered_dataframe2.loc[filtered_dataframe2['一级行业_categoryStrBig'] == '\\N', ['year', '招聘主键ID', '发布日期', 'soc_code', 'software_data', 'posting_count', '一级行业_categoryStrBig']] = np.nan
# Load the `df_samplingShare` data extracted from the yearbook
# Try different ways to weight the data: 
# 1. No weighting
# 2. weighted by the industrial share from the yearbook
# 3. weighted by the employment 
# Read the 'samplingShare' data generated from 'yearbook_data.ipynb'
df_samplingShare = pd.read_csv('D:/Dropbox/Dropbox/Paper with Yao-yu/Spatial Data Programming/Data/samplingShare.csv', encoding='utf-8')
# drop '省份_province' in df_samplingShare
df_samplingShare = df_samplingShare.drop(['省份_province'], axis=1)

# Calculate 'samplingShare' as the share of 'samplingValue' over the sum of 'samplingValue' by '城市_city' and 'year'
df_samplingShare['total_samplingValue_by_city_year'] = df_samplingShare.groupby(['城市_city', 'year'])['samplingValue'].transform('sum')
df_samplingShare['samplingShare'] = df_samplingShare['samplingValue'] / df_samplingShare['total_samplingValue_by_city_year']
# remove column 'total_samplingValue_by_city_year' 
df_samplingShare = df_samplingShare.drop(['total_samplingValue_by_city_year'], axis=1)
print(df_samplingShare)

# generate df_samplingShareTemp to only keep '城市_city'
df_samplingShareTemp = df_samplingShare[['城市_city']]
# drop duplicates in df_samplingShareTemp
df_samplingShareTemp = df_samplingShareTemp.drop_duplicates()
print(df_samplingShareTemp)
# Merge 'df_samplingShare' with 'filtered_dataframe2'
# Replace the job posting information to be null if observations only in 'filtered_dataframe2' but not in 'df_samplingShare'. Usually, this is the case that the city is not included in the yearbook.
# `Note: because we match with cities from yearbook, this action can drop some observations.`
# merge 'df_samplingShare' with 'filtered_dataframe2' on '省份_province	城市_city', 'year', '一级行业_categoryStrBig'
filtered_dataframe2 = pd.merge(df_samplingShareTemp, filtered_dataframe2, on='城市_city', how='inner')
# merge 'df_samplingShare' with 'filtered_dataframe2' on '省份_province	城市_city', 'year', '一级行业_categoryStrBig'
filtered_dataframe2 = pd.merge(df_samplingShare, filtered_dataframe2, on=['城市_city', 'year', '一级行业_categoryStrBig'], how='right')
# replace columns 'year', '招聘主键ID', '发布日期', 'soc_code', 'software_data', 'posting_count', '一级行业_categoryStrBig' with null if 'samplingShare' is null
#filtered_dataframe2.loc[filtered_dataframe2['samplingShare'].isnull(), ['year', '招聘主键ID', '发布日期', 'soc_code', 'software_data', 'posting_count', '一级行业_categoryStrBig']] = np.nan
filtered_dataframe2.head(5)
# This code block is used to convert the 2011 version industrial classification to the 2017 version. And, correct errors for `一级行业_categoryStrBig`

filtered_dataframe2['一级行业_categoryStrBig'] = filtered_dataframe2['一级行业_categoryStrBig'].replace(['房地', '批发', '批发和'], ['房地产业', '批发和零售业', '批发和零售业'])
# replace values in '二级行业_categoryStrMiddle': '广播、电视、电影和影视录音制作业' -> '广播、电视、电影和录音业', '建筑装饰和其他建筑业' -> '建筑装饰、装修和其他建筑业', 
# '仓储业' -> '装卸搬运和仓储业', '装卸搬运和运输代理业' -> '多式联运和运输代理业', '农、林、牧、渔服务业' -> '农、林、牧、渔专业及辅助性活动', 
# '开采辅助活动' -> '开采专业及辅助性活动', '石油加工、炼焦和核燃料加工业' -> '石油、煤炭及其他燃料加工业'
filtered_dataframe2['二级行业_categoryStrMiddle'] = filtered_dataframe2['二级行业_categoryStrMiddle'].replace(['广播、电视、电影和影视录音制作业', '建筑装饰和其他建筑业', '仓储业', '装卸搬运和运输代理业', 
                                                '农、林、牧、渔服务业', '开采辅助活动', '石油加工、炼焦和核燃料加工业'], ['广播、电视、电影和录音业', '建筑装饰、装修和其他建筑业', '装卸搬运和仓储业', 
                                                '多式联运和运输代理业', '农、林、牧、渔专业及辅助性活动', '开采专业及辅助性活动', '石油、煤炭及其他燃料加工业'])
# load xlsx file 'Industry_2017' from 'G:\Data' folder. The raw classification follows `GB/T 4754-2011`
dfInd2 = pd.read_excel('G:\Data\Industry_2017.xlsx', sheet_name='Sheet1')
# merge with 'dfInd1_t' on '二级行业_categoryStrMiddle', keep all observations in 'dfInd1_t'
filtered_dataframe2 = pd.merge(filtered_dataframe2, dfInd2, how='left', on='二级行业_categoryStrMiddle')
print(filtered_dataframe2.shape)
print(filtered_dataframe2.head(10))
### Generate subsamples based on the firm's type: branch, parent, and local
## subset 'resampled_df' according to 'branch', 'parent', and 'local'

# dataframe dfBranch is subset of resampled_df with 'branch' == 1
dfBranch = filtered_dataframe2[filtered_dataframe2['branch'] == 1]
# dataframe dfParent is subset of resampled_df with 'parent' == 1
dfParent = filtered_dataframe2[filtered_dataframe2['parent'] == 1]
# dataframe dfLocal is subset of resampled_df with 'local' == 1
dfLocal = filtered_dataframe2[filtered_dataframe2['local'] == 1]